<p class="lead">Every unit before this one taught a technique. This one teaches the thing the techniques are for: a finished piece of research that somebody else can check. It is meant to be read twice — at the beginning of your thesis, to see what will be expected and in what order, and at the end, as the audit checklist you run before handing anything in. It contains the laboratory's three project tracks with a specification for each, the analysis plan you write <em>before</em> fitting a single model, a worked miniature of a QSAR project executed end to end in R on the frozen ROCK1 data, the rubric expanded into criteria a supervisor can tick, and the deliverables with a realistic timeline. The miniature is deliberately small: its purpose is not to be a good model but to show the whole arc of a defensible project in one place, including the two points where it would have been easy to lie.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad cierra el curso con el proyecto final. Define las tres vías del laboratorio — QSAR y cribado virtual basado en ligandos, análisis de simulaciones y datos biomédicos — cada una con su especificación de datos, validación y entregables, e introduce el plan de análisis escrito <em>antes</em> de modelar y los tres puntos de control con el supervisor. Luego ejecuta en R una miniatura completa de la vía 1 sobre los datos congelados de ROCK1: procedencia verificada con SHA-256, partición por andamiaje decidida de antemano, líneas base, un modelo ajustado con <code>ranger</code>, el error con intervalo bootstrap, dos modos de fallo exhibidos, dominio de aplicabilidad, lista priorizada, figura de publicación y ficha del modelo. Termina con la rúbrica detallada, una auditoría ejecutable y el calendario de entregas.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Carry a project from question to repository: provenance, documented curation, baseline and tuned analyses, honest validation, uncertainty, interpretation, publication figures, a Methods section, tested code and a DOI.</li>
<li>Write a pre-registration-style analysis plan before modelling, and recognise which later decisions the plan forbids you to make.</li>
<li>Specify your own project inside one of the three tracks, with its data, its unit of prediction, its validation design and its deliverable claim.</li>
<li>Defend every number in your report against the checklists of Blocks C–F, and read each one against the noise floor of the data.</li>
<li>Score a submission with the laboratory's rubric, and run the audit checklist on your own work before anybody else does.</li>
<li>Present the work in a ten-minute lab-meeting talk built from the figures the project produced.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read Sections 1–3 before starting your project and Sections 5–7 before submitting it; run Section 4 once, line by line. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","ranger","glmnet","digest","jsonlite","patchwork","testthat"))</code> once before the first chunk; <code>Matrix</code> and <code>ggplot2</code> ship with the runtime. The whole document knits in about three minutes on four threads; every method in the miniature was taught in an earlier unit.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(Matrix);  library(ranger); library(glmnet)
  library(digest);     library(jsonlite); library(ggplot2); library(patchwork)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10), panel.grid.minor = element_blank()))
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
dp <- function(f) data_file(f)           # the course's frozen data directory
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| ranger", as.character(packageVersion("ranger")),
    "| glmnet", as.character(packageVersion("glmnet")),
    "| digest", as.character(packageVersion("digest")),
    "| ggplot2", as.character(packageVersion("ggplot2")), "\n")
cat("threads used by ranger below: 4 | global seed: 0\n")



The versions printed above are part of the result. A capstone that reports a number without the version of the library that produced it is asking its reader to take the number on trust, and the reproducibility literature of the last decade is a long argument against doing that<sup class="c"><a href="#r1">[1]</a></sup><sup class="c"><a href="#r5">[5]</a></sup>. R makes this easy — `R.version` and `packageVersion()` are always available, and `sessionInfo()` prints the complete state — and the language itself<sup class="c"><a href="#r30">[30]</a></sup>, the tidyverse<sup class="c"><a href="#r31">[31]</a></sup> and the document format that knits this page<sup class="c"><a href="#r35">[35]</a></sup> are citable objects you should name in your Methods.

## 1. The capstone is the project, not the notebook

A capstone is not a bigger exercise. The exercises in Units 1–26 each isolated one decision; a project is the same decisions taken together, under time pressure, with nobody saying which one matters. What changes is the unit of judgement: an examiner does not grade a chunk, they grade a **claim** — "this model predicts ROCK1 potency for new chemotypes to within about 0.8 log units", "these 30 genes separate diseased from healthy tissue in an independent cohort". Everything here exists to make a claim of that shape survive a sceptical reader.

Two findings from the wider literature set the standard. The first is that most computational results are not reproducible by default: in Nature's survey of 1,576 researchers, more than 70 % had failed to reproduce another scientist's experiment and more than half had failed to reproduce their own<sup class="c"><a href="#r6">[6]</a></sup>, and the diagnosis is almost always that the data, the code or the environment were never captured<sup class="c"><a href="#r5">[5]</a></sup><sup class="c"><a href="#r3">[3]</a></sup>. The second is that in machine-learning-based science the dominant error is not a bug but a **leak**: Kapoor and Narayanan found data leakage in 294 papers across 17 fields, each reporting performance that evaporated once it was closed<sup class="c"><a href="#r7">[7]</a></sup>. The remedy for both is procedural: fix the data, write the plan, choose the validation design before you see the scores, and publish everything somebody else would need to repeat the work<sup class="c"><a href="#r1">[1]</a></sup><sup class="c"><a href="#r2">[2]</a></sup>.

<div class="box bridge"><h4>Domain context — what the laboratory's projects actually look like</h4>
<p>The Laboratorio de Bioinformática y Química Computacional works on a small number of protein targets — kinases such as ROCK1, receptors such as P2X3 and the apelin receptor, protein–protein interfaces such as MDM2/MDM4, bacterial efflux pumps — and on biomedical datasets from collaborating clinical groups. A thesis in the group has one of three shapes. <strong>Ligand-based work</strong> starts from a table of measured activities: each row is a molecule, the column of interest is a potency such as an IC<sub>50</sub> (the concentration that halves a biological response) or a K<sub>i</sub> (a binding constant), and the goal is a model that ranks untested molecules. <strong>Simulation work</strong> starts from molecular-dynamics trajectories: a physics engine integrates the motion of a protein and its water, and what reaches the data scientist is a set of time series — distances, angles, root-mean-square deviations — from several independent repetitions called replicas. <strong>Biomedical work</strong> starts from a matrix of measurements on patients or samples, typically gene-expression values from a public repository, with samples in columns and genes in rows. The three kinds of data are utterly different; the project discipline around them is identical, which is why one capstone unit serves all three.</p></div>

The three tracks below are three instances of one pipeline, drawn in Figure 1: a spine of eleven stages every project walks, with the track-specific content of four of them underneath.



In [ ]:
stage <- c("question\n+ plan", "data\n+ provenance", "curation\n+ data card", "EDA", "split chosen\nin advance",
           "baseline", "model\n+ tuning", "honest metric\n+ interval", "interpretation\n+ AD",
           "figures\n+ Methods", "repo\n+ DOI")
op <- par(mar = c(0, 0, 0, 0)); plot(NA, xlim = c(-0.6, 10.2), ylim = c(-0.2, 10), axes = FALSE, xlab = "", ylab = "")
w <- 10/length(stage)
for (i in seq_along(stage)) {                                   # the shared spine, as chevrons
  x <- (i - 1)*w
  polygon(c(x, x + w*0.86, x + w*0.99, x + w*0.86, x, x + w*0.13), c(8.3, 8.3, 8.75, 9.2, 9.2, 8.75),
          col = ifelse(i %% 2 == 1, "#dce6f6", "#eaf0fa"), border = PAL[["blue"]], lwd = 0.8)
  text(x + w*0.5, 8.75, stage[i], cex = 0.44)
}
text(5, 9.65, "The spine every capstone shares", font = 2, cex = 0.72, col = "#24406e")
lanes <- list(list("Track 1 - QSAR / ligand-based virtual screening", PAL[["orange"]],
                   c("ChEMBL target of your choice\n(activities, assays, documents)", "curated compound table ->\nfrozen Morgan fingerprints",
                     "scaffold split, random forest,\nbootstrap interval, AD", "a prioritised purchasable list\n+ expected hit rate")),
              list("Track 2 - Simulation analytics", PAL[["green"]],
                   c("replica trajectories or their\nobservable tables", "equilibration detection,\nblock averaging",
                     "autocorrelation, replica-level\nuncertainty, clustering", "a defensible difference\nbetween two systems")),
              list("Track 3 - Biomedical data", PAL[["purple"]],
                   c("a public omics / clinical set\n(GEO, TCGA, MIMIC...)", "curated matrix + sample\nmetadata, batch structure",
                     "differential analysis or model\nwith external validation", "an interpreted gene set\nor a validated predictor")))
for (j in seq_along(lanes)) {
  yl <- 6.6 - (j - 1)*2.3; col <- lanes[[j]][[2]]
  text(-0.55, yl + 0.5, lanes[[j]][[1]], adj = 0, font = 2, cex = 0.55, col = col)
  for (k in 1:4) {
    x <- (k - 1)*2.5 + 0.05
    rect(x, yl - 1.05, x + 2.35, yl + 0.2, border = col, lwd = 0.9)
    text(x + 1.17, yl - 0.42, lanes[[j]][[3]][k], cex = 0.43)
    if (k < 4) arrows(x + 2.36, yl - 0.42, x + 2.48, yl - 0.42, length = 0.04, col = col, lwd = 0.9)
  }
}
par(op)



<p class="caption"><b>Figure 1.</b> The capstone pipeline. The eleven chevrons are the stages every project passes through in order; the three lanes show what four of those stages contain in each track. Nothing in the spine is optional, and the arrows are one-way on purpose — changing the split after seeing a score invalidates the rest of the diagram.</p>

## 2. The three tracks

Each track is a specification you can hand to a supervisor; the rows are the same in all three because the rubric is. Choose one and stay in it — a project that tries two has half a validation design for each.

### 2.1 Track 1 — QSAR and ligand-based virtual screening

**The question.** "Can we rank untested molecules for target X well enough to choose which fifty to buy or synthesise?" Target X is yours to choose: a ChEMBL target with at least a few hundred measured compounds that the group has a reason to care about.

| element | specification |
|---|---|
| raw data | all `IC50`/`Ki`/`Kd` activities from the ChEMBL API<sup class="c"><a href="#r24">[24]</a></sup>, downloaded once, frozen, hashed, with query and release recorded |
| curation | the Unit 6 protocol: standardisation, unit harmonisation, censored records flagged, replicate aggregation, an exclusion log |
| unit of prediction | one standardised compound, not one measurement |
| target variable | pActivity = −log<sub>10</sub>(activity in M), median over exact measurements |
| features | Morgan fingerprints (radius 2, 2,048 bits)<sup class="c"><a href="#r19">[19]</a></sup> or descriptors (Section 4.3) |
| validation | a Bemis–Murcko scaffold<sup class="c"><a href="#r18">[18]</a></sup> or leave-cluster-out split as the headline; a random split only as a contrast<sup class="c"><a href="#r15">[15]</a></sup><sup class="c"><a href="#r16">[16]</a></sup> |
| primary metric | RMSE in log units with a bootstrap 95 % interval; Spearman ρ as a ranking check |
| uncertainty & domain | nearest-neighbour similarity or conformal intervals; an explicit domain rule<sup class="c"><a href="#r22">[22]</a></sup><sup class="c"><a href="#r23">[23]</a></sup> |
| deliverable claim | N prioritised candidates with a prediction, an interval and an in-domain flag, plus the expected hit rate |

**What makes it fail.** A random split quoted as if it answered the new-chemotype question; duplicate structures on both sides of the split; censored values treated as exact; a prioritised list outside the applicability domain. Sections 4.3, 4.7 and 4.8 exhibit three of these four.

### 2.2 Track 2 — Simulation analytics

**The question.** "Do two systems — two temperatures, a wild type and a mutant, an apo and a holo form — differ in a property we can measure from the trajectories, by more than the sampling error of our own simulations?"

| element | specification |
|---|---|
| raw data | replica trajectories, or the frozen observable tables (`md_trpcage_observables.csv`: two conditions × three replicas × 500 frames), protocol recorded |
| curation | equilibration detection and discard, frame rate and units documented, a per-replica quality table<sup class="c"><a href="#r27">[27]</a></sup> |
| unit of prediction | one **replica**, never one frame — successive frames are not independent |
| target variable | the observable the question is about (helix fraction, a distance, a contact count) |
| features | time series and their summaries; principal components of the coordinates |
| validation | between-replica variance as the error bar; block averaging and autocorrelation times<sup class="c"><a href="#r27">[27]</a></sup> |
| primary metric | the between-condition difference with an interval across replicas, plus an effect size |
| uncertainty & domain | what the simulation length can and cannot resolve; a convergence plot per claim |
| deliverable claim | "property P differs by Δ (95 % CI …), n = 3 replicas per condition", or an honest "our sampling cannot resolve it" |

**What makes it fail.** Treating 500 correlated frames as 500 observations, which shrinks every interval by an order of magnitude; one replica per condition; a disagreeing replica quietly discarded. The LiveCoMS article on sampling quality is this track's checklist<sup class="c"><a href="#r27">[27]</a></sup>.

### 2.3 Track 3 — Biomedical data

**The question.** "Which features separate the two clinical groups, and does that separation hold in a cohort we did not use to find it?"

| element | specification |
|---|---|
| raw data | a public omics or clinical dataset (the course ships `geo_periodontitis_expr.csv`: 5,000 genes × 138 samples, 69 patient pairs), accession and platform recorded |
| curation | metadata reconciled with the matrix, duplicated patients identified, batch structure described, a sample-exclusion log |
| unit of prediction | one **patient**, not one sample — paired samples are not independent |
| target variable | disease status, or a continuous clinical score |
| features | genes, proteins or clinical variables; all filtering inside the folds |
| validation | patient-grouped cross-validation plus external validation when a second cohort exists<sup class="c"><a href="#r12">[12]</a></sup>; FDR control<sup class="c"><a href="#r28">[28]</a></sup> |
| primary metric | AUC with an interval, or differential genes at a stated FDR, with effect sizes |
| uncertainty & domain | calibration; the population the model applies to; a sample-size justification |
| deliverable claim | "a k-gene signature separates diseased from healthy tissue with AUC … in an independent cohort", with its interpretation |

**What makes it fail.** Feature selection on the whole matrix before splitting — the classic leak<sup class="c"><a href="#r7">[7]</a></sup>; two samples from one patient split across train and test; a claim about patients supported by an analysis of samples. TRIPOD+AI is the standard your Methods must satisfy<sup class="c"><a href="#r12">[12]</a></sup>, and DOME its counterpart for biological machine learning<sup class="c"><a href="#r11">[11]</a></sup>.

<div class="exercise"><h4>Exercise 1</h4>
<p>Write the table of your own project — the same eleven rows — before you read Section 4. Be specific: name the target or the dataset, the unit of prediction, the split, the metric and the sentence you intend to be able to write at the end. If you cannot fill the "deliverable claim" row, you do not yet have a project; you have a dataset.</p></div>

## 3. The analysis plan, written before the modelling

The plan is a short document, shown to your supervisor **before any model is fitted**, that commits you to the decisions a later you would be tempted to make after seeing the scores. It is a lightweight pre-registration, adopted for the same reason: a study whose analysis is specified in advance cannot silently become a different study once the results are in<sup class="c"><a href="#r8">[8]</a></sup>. The failure it prevents is researcher degrees of freedom — the many small, individually defensible choices whose total is a result that is not real<sup class="c"><a href="#r9">[9]</a></sup> — whose measured consequence in computational research is over-optimistic reporting<sup class="c"><a href="#r10">[10]</a></sup>.

### 3.1 The template

Fill in these nine fields. Half a page is enough; two pages means you are designing the paper rather than the analysis.

1. **Question.** One sentence, answerable with the data you have already frozen.
2. **Target variable and its units**, including the transformation (log, z-score) and why.
3. **Unit of prediction.** The row of your final table: a compound, a replica, a patient. Grouping and leakage both follow from this line.
4. **Data and its provenance.** Source, version or accession, download date, licence, hash, and what you will exclude with what rule.
5. **Split strategy, chosen now.** Which split, which grouping variable, what future it simulates, and which sensitivity analyses are contrasts.
6. **Primary metric and baselines.** One metric with its interval, the baselines it must beat (always including a trivial one), and the noise floor.
7. **Decision rule.** What result would make you say the model works — and what result would make you say it does not. The second clause is what makes the first meaningful.
8. **Planned figures.** Three to six, by name; anything added later needs a reason.
9. **Deviations.** A log of every departure from the plan with its justification.

The plan is a file in the repository, not an email. Written as a structured R list it becomes checkable, as the chunk below shows for the miniature of Section 4 — which lets the document assert that what it did matches what it said it would do. `jsonlite::toJSON()` turns the same object into the `docs/analysis_plan.json` that travels with the code.



In [ ]:
PLAN <- list(
  question = "How well does a fingerprint model predict ROCK1 potency for chemotypes it has not seen?",
  target   = "pActivity = -log10(activity in M), median over exact measurements",
  unit_of_prediction = "one standardised compound (InChIKey), not one measurement",
  data  = list(file = "rock1_chembl_curated.csv", source = "ChEMBL_37 via the EBI REST API",
               exclusions = "censored-only compounds (relation > or <) removed from the regression set"),
  split = list(primary = "Bemis-Murcko scaffold split, 80/20, largest families to training",
               simulates = "prediction for chemotypes absent from the training data",
               sensitivity = "the same pipeline under a random split, reported as a contrast only"),
  primary_metric = "RMSE in log units on the scaffold hold-out, with a 95% bootstrap interval",
  baselines = c("mean predictor", "ridge on fingerprints (glmnet)", "untuned random forest (ranger)",
                "noise floor sigma ~ 0.51 log units (Unit 6)"),
  decision_rule = paste("the model is useful if its hold-out RMSE interval lies entirely below the mean predictor",
                        "AND the top-20 prioritised list is enriched in measured actives; it is NOT useful if the",
                        "interval overlaps the mean predictor or if the candidates fall outside the domain"),
  planned_figures = c("split diagnostic", "tuning curve", "metric with interval",
                      "error vs distance to training", "predicted vs observed (publication figure)"),
  deviations = character(0))
cat(substr(toJSON(PLAN, pretty = TRUE, auto_unbox = TRUE), 1, 900), "...\n")
cat("\nfields:", length(PLAN), "| planned figures:", length(PLAN$planned_figures),
    "| baselines:", length(PLAN$baselines), "| deviations logged so far:", length(PLAN$deviations), "\n")



The printed plan has nine fields and names five figures and four baselines before a single row of data has been read. Two of its clauses do real work later. The `split$primary` clause means that when Section 4.7 computes a prettier number under a random split, that number cannot become the headline — the plan already assigned it the role of a contrast. The `decision_rule` clause contains its own refutation, so Section 4.6 cannot quietly redefine success after seeing the interval.

### 3.2 The three supervisor checkpoints

Three short meetings, each with one artefact, because the mistakes that cost a thesis a semester are made early and are cheap to catch.

| checkpoint | when | artefact you bring | what the supervisor checks |
|---|---|---|---|
| **C1 — data card** | week 4 | one page: source, version, hash, counts, exclusion log, known defects, licence | that the data are frozen, the exclusions are rules, and the unit of prediction is correct |
| **C2 — validation plan** | week 8 | the plan, the split code with its diagnostic figure, the baselines already run | that the split matches the claim, a trivial baseline exists, and nothing has been tuned on the hold-out |
| **C3 — figure draft** | week 16 | the planned figures from real results, with captions written | that each figure answers one question and nothing in it contradicts the Methods |

Figure 2 places these checkpoints on the nine-stage timeline of Section 7. Notice that C2 falls before any tuned model exists: by design, the validation design is approved while it is still cheap to change.



In [ ]:
stages <- data.frame(
  name = c("S1  Question, plan, frozen data", "S2  Curation + data card", "S3  EDA + quality report",
           "S4  Split + baseline", "S5  Model + tuning", "S6  Validation + uncertainty",
           "S7  Interpretation + domain", "S8  Figures + Methods", "S9  Repo, tests, DOI, talk"),
  start = seq(0, 16, by = 2))
stages$row <- rev(seq_len(nrow(stages)))
chk <- data.frame(week = c(4, 8, 16), lab = c("C1 data card", "C2 validation plan", "C3 figure draft"))
g <- ggplot(stages) +
  geom_rect(aes(xmin = start, xmax = start + 2, ymin = row - 0.3, ymax = row + 0.3, fill = row),
            colour = PAL[["blue"]], linewidth = 0.3, show.legend = FALSE) +
  geom_text(aes(start + 0.1, row, label = name), hjust = 0, size = 2.5) +
  geom_vline(data = chk, aes(xintercept = week), colour = PAL[["orange"]], linetype = 2, linewidth = 0.5) +
  geom_text(data = chk, aes(week, 9.9, label = lab), colour = PAL[["orange"]], size = 2.5, fontface = "bold") +
  scale_fill_gradient(low = "#c9d8ef", high = "#5b85c0") +
  scale_x_continuous(breaks = seq(0, 18, 2), limits = c(0, 19.5)) +
  scale_y_continuous(limits = c(0.2, 10.3)) +
  labs(x = "week of the project", y = NULL,
       title = "A nine-stage capstone: two weeks per stage, three supervisor checkpoints") +
  theme(axis.text.y = element_blank(), panel.grid.major.y = element_blank())
print(g)
cat("stages:", nrow(stages), "| nominal duration:", 2*nrow(stages), "weeks | checkpoints at weeks",
    paste(chk$week, collapse = ", "), "\n")



<p class="caption"><b>Figure 2.</b> The capstone schedule. Nine stages of two weeks, eighteen weeks in total, with the checkpoints in orange at weeks 4, 8 and 16. The ordering is the point: the data card is approved before any analysis, the validation plan before any tuned model, the figures before the text explaining them.</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Take the plan of Exercise 1 and write its three checkpoint artefacts as empty files in your repository now: <code>docs/data_card.md</code>, <code>docs/analysis_plan.md</code> and <code>figures/README.md</code>. Projects that create these files on day one finish them; projects that intend to write them at the end do not.</p></div>

## 4. A worked miniature of Track 1, end to end

The rest of this section executes a complete, deliberately tiny Track 1 project on the frozen ROCK1 snapshot. It is not a contribution to the ROCK1 literature: the sibling units hold the depth — Unit 6 the curation protocol, Unit 11 the leakage-free pipeline, Unit 12 the splits and metrics, Unit 14 the tuning, Unit 17 the applicability domain, Unit 23 the publication figure. What is new here is the **order**, the **provenance** and the **audit**.

### 4.1 Provenance: the hash comes first

Before anything is analysed, the files are pinned. A hash of each raw file, stored with the query that produced it and the download date, converts "the ChEMBL data" into a specific object comparable byte for byte years later — the minimum content of the F in FAIR<sup class="c"><a href="#r4">[4]</a></sup> and the first line of every reproducibility checklist<sup class="c"><a href="#r1">[1]</a></sup><sup class="c"><a href="#r2">[2]</a></sup>. In R this is one call: `digest::digest(file = path, algo = "sha256")` returns the same hexadecimal string that `sha256sum` or Python's `hashlib` would.



In [ ]:
man  <- fromJSON(dp("manifest_rock1.json"))
FILE <- "rock1_chembl_curated.csv"
sha      <- digest(file = dp(FILE), algo = "sha256")
expected <- man$files[[FILE]]$sha256
HASH_OK  <- identical(sha, expected)
cat("file           :", FILE, "\n")
cat("sha256 (now)   :", substr(sha, 1, 32), "...\n")
cat("sha256 (frozen):", substr(expected, 1, 32), "...\n")
cat("match          :", HASH_OK, "\n")
cat("source         :", man$source, "|", man$chembl_release, "released", man$chembl_release_date,
    "| downloaded", man$download_date, "\n")
cat("licence        :", man$licence, "\n")
cat("query          :", man$query, "\n")
cat("recorded size  :", man$files[[FILE]]$n_rows, "rows x", man$files[[FILE]]$n_cols, "columns\n")



The hash matches, so every number below refers to an unambiguously identified file. In your own project this check belongs in a test (Section 4.11), not in a chunk you might forget to run: a silent change in an input makes every downstream number wrong without anything looking broken.

<div class="box bridge"><h4>Domain context — what a ChEMBL activity record is</h4>
<p>ChEMBL is the European public repository of measured bioactivities, extracted by curators from the medicinal-chemistry literature<sup class="c"><a href="#r24">[24]</a></sup>. One record is one measurement of one compound against one target in one assay, from one publication. The pitfalls that make curation necessary are all in the raw file of this course: the same compound measured in twelve assays; three incomparable measurement types (IC<sub>50</sub>, K<sub>i</sub>, K<sub>d</sub>) in one column; <strong>censored</strong> values, where the experiment established only that the activity is weaker than the highest concentration tested (<code>&gt; 10000 nM</code>); units differing by nine orders of magnitude; duplicate deposits of one paper. The curated table aggregates to one row per standardised parent structure and keeps censored compounds flagged rather than deleting them, because excluding them silently biases the potency distribution upwards.</p></div>

### 4.2 Curation reuse and the data card

The capstone does not re-invent curation: it reuses a documented protocol and reports its exclusion log. That log is the body of the data card — ten steps, each with the rows and molecules that survived.



In [ ]:
log_ <- fread(dp("rock1_curation_log.csv"))
df   <- fread(dp(FILE))
print(log_[, .(step, description, n_rows, n_molecules)], nrows = 10, class = FALSE)
cat("\ncurated table:", nrow(df), "compounds x", ncol(df), "columns\n")
cat("censored-only compounds (excluded from the regression target):", sum(df$censored), "\n")
cat("compounds with at least one exact measurement:", sum(!df$censored), "\n")
cat("compounds whose replicates span more than one log unit (wide_spread):", sum(df$wide_spread), "\n")


In [ ]:
funnel <- rbind(data.frame(step = log_$step, n = log_$n_rows, what = "records"),
                data.frame(step = log_$step, n = log_$n_molecules, what = "distinct molecules"))
g <- ggplot(funnel, aes(factor(step, levels = rev(log_$step)), n, fill = what)) +
  geom_col(data = subset(funnel, what == "records"), width = 0.75) +
  geom_col(data = subset(funnel, what == "distinct molecules"), width = 0.38) +
  geom_text(data = subset(funnel, what == "records"), aes(label = format(n, big.mark = ",")),
            hjust = -0.12, size = 2.3) +
  coord_flip() + scale_y_continuous(limits = c(0, 6200)) +
  scale_fill_manual(values = c(records = "#9cb8e4", `distinct molecules` = unname(PAL["orange"])), name = NULL) +
  labs(x = "curation step", y = "rows / molecules surviving",
       title = "The exclusion log of the frozen ROCK1 snapshot") +
  theme(legend.position = "bottom")
print(g)
cat("rows removed by the ten steps:", log_$n_rows[1] - log_$n_rows[nrow(log_)],
    sprintf("(%.1f%% of the raw records)\n", 100*(1 - log_$n_rows[nrow(log_)]/log_$n_rows[1])))



<p class="caption"><b>Figure 3.</b> The curation funnel. Of 5,089 raw activity records, 5,045 survive the ten deterministic steps, and aggregating replicates per standardised structure collapses them to 4,280 compounds — of which 2,764 carry at least one exact measurement and form the regression set. The gap between the blue bar (records) and the orange bar (distinct molecules) is the replicate structure that makes compound-level aggregation necessary; the ten steps remove 809 rows, 15.9 % of the raw file, the largest single cut being the final aggregation.</p>

The data card adds three things the log does not contain: the licence (CC BY-SA 3.0), the defects that survive curation (mixed assay types; the 60 compounds whose replicates span more than a log unit) and the intended use — ranking untested compounds for this target, not predicting absolute potency elsewhere. That triple is what "datasheets for datasets" asks for<sup class="c"><a href="#r26">[26]</a></sup>, and it is the artefact of checkpoint C1.

### 4.3 The split, chosen before any model is fitted

The plan committed to a Bemis–Murcko scaffold split<sup class="c"><a href="#r18">[18]</a></sup>: families sorted by size, the largest to training until 80 % of the compounds are used, the rare isolated structures to the hold-out. It simulates the question the laboratory actually asks — can the model say anything about a chemotype it has not seen?<sup class="c"><a href="#r15">[15]</a></sup><sup class="c"><a href="#r16">[16]</a></sup> The random split is computed too, with the role of a contrast assigned in advance.

<div class="box bridge"><h4>Substitution — fingerprints come from a frozen file, not from RDKit</h4>
<p>The Python edition computes Morgan fingerprints<sup class="c"><a href="#r19">[19]</a></sup> in the notebook with RDKit. R has no RDKit binding here, so the fingerprints of every molecule in the course were generated once with RDKit 2026.03.4 (<code>GetMorganGenerator(radius = 2, fpSize = 2048)</code>) and frozen as sparse <code>(id, bit)</code> pairs in <code>data/rock1_morgan_sparse.csv.gz</code> and <code>data/screening_morgan_sparse.csv.gz</code>; <code>data/manifest_fingerprints.json</code> records their hashes and the generator call, and <code>data/rock1_curated.sdf.gz</code> ships the structures so a reader with ChemmineR or rcdk can recompute them. The chunk below rebuilds the 2,764 × 2,048 binary matrix with <code>Matrix::sparseMatrix()</code>; Tanimoto similarity is then sparse algebra, \( T(a,b) = |a \cap b| / (|a| + |b| - |a \cap b|) \), with the intersection from <code>tcrossprod()</code>. For a capstone this substitution belongs in the Methods, exactly as written here — never silently.</p></div>



In [ ]:
reg  <- df[censored == FALSE]
y    <- reg$pActivity
scaf <- reg$scaffold_smiles
sp   <- fread(dp("rock1_morgan_sparse.csv.gz"))            # frozen RDKit Morgan bits (id, bit)
keep <- sp[id %in% reg$inchikey]
FP   <- sparseMatrix(i = match(keep$id, reg$inchikey), j = keep$bit + 1L, x = 1,
                     dims = c(nrow(reg), 2048))
X    <- as.matrix(FP); colnames(X) <- paste0("b", 0:2047); storage.mode(X) <- "double"
scaffold_split <- function(s, test_size = 0.2) {    # deterministic: largest families to training (Unit 12)
  g <- split(seq_along(s), s)
  o <- order(-lengths(g), vapply(g, function(i) i[1], integer(1)))
  n_train <- round((1 - test_size)*length(s)); tr <- te <- integer(0)
  for (k in o) { i <- g[[k]]; if (length(tr) < n_train) tr <- c(tr, i) else te <- c(te, i) }
  list(tr = tr, te = te) }
S <- scaffold_split(scaf, 0.2); tr <- S$tr; te <- S$te
set.seed(1); perm <- sample(length(y))                      # contrast only
rtr <- perm[seq_along(tr)]; rte <- perm[(length(tr) + 1):length(y)]
cat(sprintf("compounds %d | features %d | scaffold families %d | mean set bits %.1f\n",
            length(y), ncol(FP), uniqueN(scaf), mean(rowSums(FP))))
cat(sprintf("scaffold split : train %d, test %d, shared scaffolds %d\n",
            length(tr), length(te), length(intersect(scaf[tr], scaf[te]))))
cat(sprintf("random split   : train %d, test %d, shared scaffolds %d\n",
            length(rtr), length(rte), length(intersect(scaf[rtr], scaf[rte]))))
cat(sprintf("pActivity  train %.3f +/- %.3f | scaffold test %.3f +/- %.3f | random test %.3f +/- %.3f\n",
            mean(y[tr]), sd(y[tr]), mean(y[te]), sd(y[te]), mean(y[rte]), sd(y[rte])))



The two splits differ in two ways that matter. The scaffold split shares **zero** scaffold families between the two sides, while the random split shares 175, so most of its hold-out compounds have a near-twin in training. And the scaffold hold-out is less potent than the training set (6.19 against 6.59 log units), because potent compounds accumulate in the large optimised series a size-ordered split sends to training — informative rather than broken, but it must be quoted whenever the hold-out numbers are. Figure 4 shows the first difference with the diagnostic of Unit 12.



In [ ]:
nbits <- rowSums(FP)
tanimoto_max <- function(q, ref) {                 # max Tanimoto of each query row to any reference row
  inter <- as.matrix(tcrossprod(FP[q, , drop = FALSE], FP[ref, , drop = FALSE]))
  sim   <- inter/(outer(nbits[q], nbits[ref], "+") - inter)
  apply(sim, 1, max) }
nn_scaf <- tanimoto_max(te, tr); nn_rand <- tanimoto_max(rte, rtr)
d1 <- rbind(data.frame(sim = nn_rand, split = "random hold-out"),
            data.frame(sim = nn_scaf, split = "scaffold hold-out"))
g1 <- ggplot(d1, aes(sim, fill = split)) +
  geom_histogram(bins = 30, alpha = 0.65, position = "identity") +
  geom_vline(xintercept = c(median(nn_rand), median(nn_scaf)),
             colour = c(PAL[["blue"]], PAL[["orange"]]), linetype = 2) +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "Tanimoto similarity to the nearest training compound", y = "hold-out compounds",
       title = "The scaffold split moves the hold-out away") +
  theme(legend.position = "top")
d2 <- rbind(data.frame(y = y[tr], g = "train\n(scaffold)"), data.frame(y = y[te], g = "test\n(scaffold)"),
            data.frame(y = y[rte], g = "test\n(random)"))
d2$g <- factor(d2$g, levels = c("train\n(scaffold)", "test\n(scaffold)", "test\n(random)"))
g2 <- ggplot(d2, aes(g, y)) + geom_boxplot(width = 0.55, outlier.size = 0.4,
                                           colour = PAL[["blue"]], fill = "#eaf0fa") +
  stat_summary(fun = median, geom = "point", colour = PAL[["orange"]], size = 1.6) +
  labs(x = NULL, y = "pActivity (-log10 M)", title = "and shifts the target distribution")
print(g1 | g2)
cat(sprintf("median nearest-neighbour similarity: scaffold %.3f | random %.3f\n",
            median(nn_scaf), median(nn_rand)))
cat(sprintf("scaffold hold-out compounds with similarity below 0.35: %d (%.1f%%)\n",
            sum(nn_scaf < 0.35), 100*mean(nn_scaf < 0.35)))



<p class="caption"><b>Figure 4.</b> Left: the distance from each hold-out compound to the training set. Under the random split the median nearest-neighbour Tanimoto similarity is 0.784 — most test compounds have a close analogue in training. Under the scaffold split it falls to 0.661, and 86 compounds (15.6 % of the hold-out) sit below 0.35, where fingerprint models stop interpolating. Right: the scaffold hold-out is also 0.40 log units less potent than training, while the random hold-out is only 0.10 log units below it.</p>

### 4.4 Baselines

A result without a baseline is not a result. The plan named four: a mean predictor, a linear model on the same features, an untuned random forest<sup class="c"><a href="#r20">[20]</a></sup>, and the noise floor of the data itself. The last is the most important and the most often omitted — Unit 6 estimated this dataset's experimental reproducibility from its replicate measurements at about \( \sigma \approx 0.51 \) log units, which caps the attainable \( R^2 \) on this hold-out at \( 1 - \sigma^2/\mathrm{var}(y) \), as the chunk below prints. RMSE 0.8 against a noise floor of 0.5 is not a disappointing model but one within a factor of two of the measurement, and saying so is part of honest reporting<sup class="c"><a href="#r17">[17]</a></sup><sup class="c"><a href="#r13">[13]</a></sup>. The ridge baseline is `glmnet` with `alpha = 0`<sup class="c"><a href="#r33">[33]</a></sup> and its penalty chosen by five-fold cross-validation **inside the training set** — the Python edition fixed `alpha = 10` instead, a difference of implementation, not of design.



In [ ]:
rmse <- function(a, b) sqrt(mean((a - b)^2))
r2   <- function(a, b) 1 - sum((a - b)^2)/sum((a - mean(a))^2)
NOISE_FLOOR <- 0.51                                   # log units, from the replicates (Unit 6)
pred_mean <- rep(mean(y[tr]), length(te))
set.seed(2)
cvr <- cv.glmnet(FP[tr, ], y[tr], alpha = 0, nfolds = 5)
pred_ridge <- as.numeric(predict(cvr, newx = FP[te, ], s = "lambda.min"))
rf_default <- ranger(x = X[tr, ], y = y[tr], num.trees = 500, seed = 1, num.threads = 4)
pred_default <- predict(rf_default, data = X[te, ])$predictions
base_tab <- data.frame(
  baseline = c("mean predictor", "ridge on fingerprints (glmnet)", "random forest, untuned (ranger)"),
  RMSE = c(rmse(y[te], pred_mean), rmse(y[te], pred_ridge), rmse(y[te], pred_default)),
  MAE  = c(mean(abs(y[te] - pred_mean)), mean(abs(y[te] - pred_ridge)), mean(abs(y[te] - pred_default))),
  R2   = c(r2(y[te], pred_mean), r2(y[te], pred_ridge), r2(y[te], pred_default)))
knitr::kable(base_tab, digits = 3, row.names = FALSE, caption = "Baselines on the scaffold hold-out")
cat(sprintf("\nridge penalty chosen inside the training folds: lambda.min = %.3f\n", cvr$lambda.min))
cat(sprintf("untuned forest: mtry = %d (ranger's default, floor(sqrt(p))), min.node.size = 5, 500 trees\n",
            rf_default$mtry))
cat(sprintf("noise floor (Unit 6): sigma = %.2f log units -> R2 ceiling on this hold-out = %.2f\n",
            NOISE_FLOOR, 1 - NOISE_FLOOR^2/var(y[te])))



The mean predictor scores RMSE 1.123 log units, the ridge model 0.911 and the untuned forest 0.825. Three readings follow. Fingerprints carry real signal — the forest removes about a quarter of the trivial predictor's error. The nonlinear model beats the linear one on the same features by 0.09 log units, which is the usual ordering on fingerprint data. And the gap between the forest and the noise floor, 0.825 against 0.51, says that roughly half the variance a perfect model could explain is still unexplained: the \( R^2 \) ceiling set by the measurement is 0.76, and the untuned forest reaches 0.38.

### 4.5 One tuned model, tuned inside the training set

Tuning happens on the training data only, with a grouped cross-validation respecting the same scaffold structure as the outer split — otherwise the hyperparameter choice is informed by the chemotypes it will be tested on, the commonest leakage<sup class="c"><a href="#r7">[7]</a></sup>. R has no `GroupKFold`, so the lines below reproduce it: scaffold families are sorted by size and each is assigned to the fold that currently holds the fewest compounds, as scikit-learn's implementation does. The grid is deliberately nine `ranger`<sup class="c"><a href="#r32">[32]</a></sup> configurations: a small grid is defensible for a capstone, and its size belongs in the Methods.



In [ ]:
group_kfold <- function(g, k = 5) {                 # greedy balanced assignment of whole groups to folds
  tab <- sort(table(g), decreasing = TRUE); load_ <- numeric(k)
  fold_of <- setNames(integer(length(tab)), names(tab))
  for (i in seq_along(tab)) { f <- which.min(load_); fold_of[names(tab)[i]] <- f; load_[f] <- load_[f] + tab[[i]] }
  as.integer(fold_of[as.character(g)]) }
fold <- group_kfold(scaf[tr], 5)
MTRY <- c(sqrt = floor(sqrt(2048)), `0.1p` = round(0.1*2048), `0.3p` = round(0.3*2048))
grid <- expand.grid(mtry_rule = names(MTRY), min_node = c(1, 2, 5), stringsAsFactors = FALSE)
rows <- vector("list", nrow(grid))
for (i in seq_len(nrow(grid))) {
  fs <- numeric(5)
  for (f in 1:5) {
    a <- which(fold != f); b <- which(fold == f)
    m <- ranger(x = X[tr[a], ], y = y[tr[a]], num.trees = 100, mtry = MTRY[[grid$mtry_rule[i]]],
                min.node.size = grid$min_node[i], seed = 1, num.threads = 4)
    fs[f] <- rmse(y[tr[b]], predict(m, data = X[tr[b], ])$predictions) }
  rows[[i]] <- data.frame(mtry_rule = grid$mtry_rule[i], mtry = MTRY[[grid$mtry_rule[i]]],
                          min_node = grid$min_node[i], cv_rmse = mean(fs), cv_sd = sd(fs)) }
tuning <- do.call(rbind, rows); tuning <- tuning[order(tuning$cv_rmse), ]; rownames(tuning) <- NULL
best <- tuning[1, ]
knitr::kable(tuning, digits = 4, row.names = FALSE,
             caption = "Nine configurations scored by scaffold-grouped 5-fold CV on the training set only")
cat(sprintf("\nfold sizes: %s\n", paste(table(fold), collapse = " / ")))
cat(sprintf("best configuration: mtry = %d (%s), min.node.size = %d (inner CV RMSE %.4f +/- %.4f)\n",
            best$mtry, best$mtry_rule, best$min_node, best$cv_rmse, best$cv_sd))
cat(sprintf("spread across the whole grid: %.4f log units\n", max(tuning$cv_rmse) - min(tuning$cv_rmse)))


In [ ]:
tuning$label <- sprintf("mtry=%d (%s), leaf=%d", tuning$mtry, tuning$mtry_rule, tuning$min_node)
tuning$label <- factor(tuning$label, levels = rev(tuning$label))
g <- ggplot(tuning, aes(cv_rmse, label)) +
  geom_errorbarh(aes(xmin = cv_rmse - cv_sd, xmax = cv_rmse + cv_sd), height = 0.25, colour = "#9cb8e4") +
  geom_point(colour = PAL[["blue"]], size = 1.8) +
  geom_vline(xintercept = min(tuning$cv_rmse), colour = PAL[["orange"]], linetype = 2) +
  labs(x = "inner grouped 5-fold CV RMSE on the training set (log units)", y = NULL,
       title = "Nine configurations, one selected; error bars are fold-to-fold spread") +
  theme(axis.text.y = element_text(size = 6.5))
print(g)
cat(sprintf("grid span %.4f vs median within-configuration fold SD %.4f log units\n",
            max(tuning$cv_rmse) - min(tuning$cv_rmse), median(tuning$cv_sd)))



<p class="caption"><b>Figure 5.</b> The nine candidate configurations ranked by inner cross-validated RMSE, with the standard deviation across the five scaffold-grouped folds as the error bar. The whole grid spans 0.011 log units while a single configuration's folds vary by about 0.029: the differences between most candidates are smaller than the noise measuring them, and only one effect is resolvable — the three configurations with <code>mtry = 614</code> (0.3 p) are consistently the worst. A 500-point search would select a winner whose advantage this experiment cannot distinguish from luck.</p>

This is the first place where the R and the Python editions disagree in detail, and the disagreement is instructive. In the Python edition the resolvable effect was leaf size (`min_samples_leaf = 5` was clearly worse); here, with `ranger`, the large `mtry` values are the ones that hurt and leaf size is nearly irrelevant. Both runs agree on what matters — the grid span is an order of magnitude smaller than the fold-to-fold noise — and disagree about which knob is harmful, because the two forests have different splitting defaults. A capstone claiming "`mtry = 614` is harmful for kinase QSAR" from nine runs in one library would be over-reading its own experiment.

### 4.6 The honest metric and its interval

The selected configuration is refitted on the whole training set and applied once to the hold-out. Once: the hold-out is now spent, and any decision taken after seeing this number must be reported as such. A point estimate on 553 compounds is not a result by itself — it carries a sampling error that a non-parametric bootstrap quantifies<sup class="c"><a href="#r21">[21]</a></sup>.



In [ ]:
tuned <- ranger(x = X[tr, ], y = y[tr], num.trees = 500, mtry = best$mtry,
                min.node.size = best$min_node, seed = 1, num.threads = 4)
pred_tuned <- predict(tuned, data = X[te, ])$predictions
metrics <- c(rmse = rmse(y[te], pred_tuned), mae = mean(abs(y[te] - pred_tuned)),
             r2 = r2(y[te], pred_tuned), spearman = cor(y[te], pred_tuned, method = "spearman"))
cat("scaffold hold-out, tuned model: ",
    paste(sprintf("%s %.3f", names(metrics), metrics), collapse = " | "), "\n")
cat(sprintf("untuned forest for comparison : rmse %.3f\n", rmse(y[te], pred_default)))
cat(sprintf("hold-out compounds: %d | mean predictor: rmse %.3f\n", length(te), rmse(y[te], pred_mean)))


In [ ]:
B <- 2000; n_te <- length(te)
set.seed(3)
idx <- matrix(sample.int(n_te, B*n_te, replace = TRUE), nrow = B)
boot_tuned <- apply(idx, 1, function(i) rmse(y[te][i], pred_tuned[i]))
boot_diff  <- apply(idx, 1, function(i) rmse(y[te][i], pred_tuned[i]) - rmse(y[te][i], pred_default[i]))
ci      <- unname(quantile(boot_tuned, c(0.025, 0.975)))
ci_diff <- unname(quantile(boot_diff,  c(0.025, 0.975)))
cat(sprintf("tuned RMSE %.3f, 95%% bootstrap CI [%.3f, %.3f]  (B = %d resamples)\n",
            metrics[["rmse"]], ci[1], ci[2], B))
cat(sprintf("tuned - untuned: %.4f, 95%% CI [%.4f, %.4f], P(tuned better) = %.2f\n",
            metrics[["rmse"]] - rmse(y[te], pred_default), ci_diff[1], ci_diff[2], mean(boot_diff < 0)))


In [ ]:
g1 <- ggplot(data.frame(x = boot_tuned), aes(x)) +
  geom_histogram(bins = 40, fill = "#9cb8e4", colour = "white", linewidth = 0.2) +
  geom_vline(xintercept = metrics[["rmse"]], colour = PAL[["blue"]], linewidth = 0.8) +
  geom_vline(xintercept = ci, colour = PAL[["orange"]], linetype = 2) +
  geom_vline(xintercept = NOISE_FLOOR, colour = PAL[["green"]], linewidth = 0.8) +
  geom_vline(xintercept = rmse(y[te], pred_mean), colour = "black", linetype = 3) +
  annotate("text", x = NOISE_FLOOR + 0.02, y = Inf, label = "noise floor 0.51", hjust = 0, vjust = 1.5,
           size = 2.3, colour = PAL[["green"]]) +
  annotate("text", x = rmse(y[te], pred_mean) - 0.02, y = Inf, label = "mean predictor", hjust = 1,
           vjust = 1.5, size = 2.3) +
  scale_x_continuous(limits = c(0.45, 1.2)) +
  labs(x = "hold-out RMSE (log units)", y = "bootstrap resamples",
       title = "The metric is an interval, not a number")
g2 <- ggplot(data.frame(x = boot_diff), aes(x)) +
  geom_histogram(bins = 40, fill = "#f0c8c8", colour = "white", linewidth = 0.2) +
  geom_vline(xintercept = 0, colour = "black", linewidth = 0.8) +
  geom_vline(xintercept = ci_diff, colour = PAL[["orange"]], linetype = 2) +
  labs(x = "RMSE(tuned) - RMSE(untuned), paired bootstrap", y = "bootstrap resamples",
       title = "Did tuning actually help?")
print(g1 | g2)
cat(sprintf("interval width %.3f log units; the mean predictor lies %.2f interval widths above the upper limit\n",
            ci[2] - ci[1], (rmse(y[te], pred_mean) - ci[2])/(ci[2] - ci[1])))



<p class="caption"><b>Figure 6.</b> Left: the bootstrap distribution of the hold-out RMSE of the tuned model. The point estimate is 0.819 log units with a 95 % interval of [0.769, 0.872]; it lies entirely below the mean predictor (1.123, dotted) and entirely above the noise floor (0.51, green) — exactly the sandwich the decision rule asked about, with the trivial baseline sitting 2.45 interval widths above the upper limit. Right: the paired bootstrap of the difference between the tuned and the untuned forest, −0.006 log units with an interval of [−0.015, +0.003] that includes zero. The honest sentence is "tuning changed the error by less than the hold-out can resolve", even though the tuned model was better in 89 % of the resamples.</p>

That right-hand panel is the most useful habit in this unit. A capstone that reports "after hyperparameter optimisation the RMSE improved from 0.825 to 0.819" is reporting a difference it cannot measure; the honest version costs one chunk, and "better in 89 % of resamples" is not the same claim as "better".

<div class="box key"><h4>Worked example — the metric, by hand</h4>
<p>Root-mean-square error is worth computing once with a pencil so that the interval above means something concrete. Take the first five compounds of the scaffold hold-out, with the measured value <em>y</em> and the tuned model's prediction ŷ printed by the chunk below:</p>
<table>
<tr><th>compound (InChIKey prefix)</th><th>y</th><th>ŷ</th><th>e = y − ŷ</th><th>e²</th></tr>
<tr><td>JTVBXQAYBIJXRP</td><td>7.553</td><td>6.734</td><td>+0.819</td><td>0.671</td></tr>
<tr><td>JUBQGWGPNRKSSH</td><td>5.367</td><td>6.876</td><td>−1.509</td><td>2.276</td></tr>
<tr><td>JVEMVINZFPYWPW</td><td>4.267</td><td>5.670</td><td>−1.403</td><td>1.968</td></tr>
<tr><td>JWQOJVOKBAAAAR</td><td>6.700</td><td>5.961</td><td>+0.739</td><td>0.546</td></tr>
<tr><td>JXMYIDZREHNPDO</td><td>7.700</td><td>6.565</td><td>+1.135</td><td>1.288</td></tr>
</table>
<p>The columns are rounded to three decimals, which is why squaring the rounded residual differs from the printed square in the last digit. The squared errors sum to 6.749, so the mean squared error is 6.749 / 5 = 1.350 and the RMSE is \( \sqrt{1.350} = 1.162 \) log units; the mean absolute error is (0.819 + 1.509 + 1.403 + 0.739 + 1.135) / 5 = 1.121. Two lessons fall out of five rows. RMSE exceeds MAE whenever the errors are uneven, because squaring weights the two 1.4-log misses far more heavily than the 0.7-log one: RMSE is the right metric when a large error is disproportionately expensive. And these five compounds give 1.162 while the full 553-compound hold-out gives 0.819 — a five-point estimate of a model's error is worthless. The chunk below reproduces the arithmetic from the model's own predictions and stops the knit if the table above ever drifts away from it.</p></div>



In [ ]:
hand <- data.frame(inchikey = substr(reg$inchikey[te[1:5]], 1, 14),
                   y = y[te][1:5], y_hat = pred_tuned[1:5])
hand$error <- hand$y - hand$y_hat
hand$sq_error <- hand$error^2
knitr::kable(hand, digits = 3, row.names = FALSE, caption = "The five rows traced by hand above")
mse5 <- mean(hand$sq_error)
cat(sprintf("\nsum of squared errors %.3f | MSE %.3f | RMSE %.3f | MAE %.3f\n",
            sum(hand$sq_error), mse5, sqrt(mse5), mean(abs(hand$error))))
cat(sprintf("full hold-out RMSE for contrast: %.3f\n", metrics[["rmse"]]))
stopifnot(abs(sqrt(mse5) - 1.162) < 0.01, abs(mean(abs(hand$error)) - 1.121) < 0.01)
cat("hand-worked table verified against the executed model\n")



### 4.7 Two failure modes, exhibited

A capstone is judged partly on the mistakes it did **not** make, so it is worth seeing two happen. Both are available to this project at this moment, both would improve the headline number, and both are invisible in a finished report unless disclosed. The first is **choosing the protocol after seeing the scores**: the same pipeline under the random split produces a smaller error, and nothing in the code marks which number belongs to which claim. The second is **selecting the model on the hold-out** — with nine configurations one can simply report the best test score. With a grid this small the effect is tiny, which is itself the lesson: the bias grows with the number of configurations tried, and the only defence that scales is never to look.



In [ ]:
m_rand <- ranger(x = X[rtr, ], y = y[rtr], num.trees = 500, mtry = best$mtry,
                 min.node.size = best$min_node, seed = 1, num.threads = 4)
rmse_rand <- rmse(y[rte], predict(m_rand, data = X[rte, ])$predictions)
cat(sprintf("same pipeline, random hold-out   : RMSE %.3f\n", rmse_rand))
cat(sprintf("same pipeline, scaffold hold-out : RMSE %.3f\n", metrics[["rmse"]]))
cat(sprintf("optimism of the easier protocol  : %.3f log units (%.0f%% of the scaffold error)\n",
            metrics[["rmse"]] - rmse_rand, 100*(metrics[["rmse"]] - rmse_rand)/metrics[["rmse"]]))


In [ ]:
test_scores <- numeric(nrow(grid))
for (i in seq_len(nrow(grid))) {
  m <- ranger(x = X[tr, ], y = y[tr], num.trees = 100, mtry = MTRY[[grid$mtry_rule[i]]],
              min.node.size = grid$min_node[i], seed = 1, num.threads = 4)
  test_scores[i] <- rmse(y[te], predict(m, data = X[te, ])$predictions) }
sel <- which(grid$mtry_rule == best$mtry_rule & grid$min_node == best$min_node)
win <- which.min(test_scores)
cat(sprintf("best-of-grid measured ON THE HOLD-OUT : RMSE %.3f (mtry %s, leaf %d)\n",
            test_scores[win], grid$mtry_rule[win], grid$min_node[win]))
cat(sprintf("configuration selected honestly       : RMSE %.3f (mtry %s, leaf %d)\n",
            test_scores[sel], grid$mtry_rule[sel], grid$min_node[sel]))
cat(sprintf("winner's curse with %d configurations  : %.4f log units\n",
            nrow(grid), test_scores[sel] - test_scores[win]))



The random split returns 0.725 against the scaffold split's 0.819: 12 % of the error disappears purely by changing which future is simulated, and on larger, more redundant series the same substitution is worth more still<sup class="c"><a href="#r15">[15]</a></sup><sup class="c"><a href="#r16">[16]</a></sup>. Selecting the configuration on the hold-out buys another 0.007 log units — negligible, because nine candidates is a small lottery. Run five hundred, or re-run the search after every disappointing result, and the same mechanism produces the published models that do not reproduce<sup class="c"><a href="#r7">[7]</a></sup><sup class="c"><a href="#r10">[10]</a></sup>. Both numbers were computed *and reported*: the plan's sensitivity clause allows the contrast to exist, provided the headline stays with the scaffold number.

### 4.8 Applicability domain: where the model is allowed to speak

A regression model returns a number for every molecule it is shown, including molecules resembling nothing it has seen. The applicability domain is the explicit rule deciding which of those numbers you will quote<sup class="c"><a href="#r22">[22]</a></sup><sup class="c"><a href="#r23">[23]</a></sup>, and the OECD validation principles make having such a rule a requirement, not a refinement<sup class="c"><a href="#r14">[14]</a></sup>. The simplest defensible rule for a fingerprint model — the one Unit 17 developed into conformal prediction — is similarity to the nearest training compound.



In [ ]:
err <- abs(y[te] - pred_tuned)
edges <- c(0, 0.3, 0.4, 0.5, 1.01)
ad_tab <- do.call(rbind, lapply(1:(length(edges) - 1), function(j) {
  s <- nn_scaf >= edges[j] & nn_scaf < edges[j + 1]
  data.frame(similarity = sprintf("[%.2f, %.2f)", edges[j], edges[j + 1]), n = sum(s),
             MAE = mean(err[s]), RMSE = rmse(y[te][s], pred_tuned[s])) }))
knitr::kable(ad_tab, digits = 3, row.names = FALSE,
             caption = "Hold-out error by distance to the training set")
ord <- order(-nn_scaf); fracs <- seq(0.1, 1, by = 0.05)
cov_rmse <- vapply(fracs, function(f) { k <- ord[1:floor(f*n_te)]; rmse(y[te][k], pred_tuned[k]) }, numeric(1))
cat(sprintf("\nRMSE on the 20%% most similar hold-out compounds: %.3f | on all of them: %.3f\n",
            cov_rmse[3], metrics[["rmse"]]))
cat(sprintf("compounds in the least similar band: %d (%.1f%% of the hold-out)\n",
            ad_tab$n[1], 100*ad_tab$n[1]/n_te))


In [ ]:
g1 <- ggplot(ad_tab, aes(similarity, RMSE)) +
  geom_col(fill = "#9cb8e4", colour = PAL[["blue"]], width = 0.7) +
  geom_text(aes(label = paste0("n=", n)), vjust = -0.5, size = 2.4) +
  geom_hline(yintercept = metrics[["rmse"]], colour = PAL[["orange"]], linetype = 2) +
  geom_hline(yintercept = NOISE_FLOOR, colour = PAL[["green"]], linewidth = 0.7) +
  scale_y_continuous(limits = c(0, 1.2)) +
  labs(x = "Tanimoto similarity to nearest training compound", y = "RMSE (log units)",
       title = "Error grows as the model extrapolates")
g2 <- ggplot(data.frame(cov = 100*fracs, rmse = cov_rmse), aes(cov, rmse)) +
  geom_line(colour = PAL[["blue"]]) + geom_point(colour = PAL[["blue"]], size = 1.4) +
  geom_hline(yintercept = metrics[["rmse"]], colour = PAL[["orange"]], linetype = 2) +
  labs(x = "coverage: % of the hold-out retained, most similar first",
       y = "RMSE of the retained set", title = "Coverage bought with accuracy")
print(g1 | g2)
cat(sprintf("error ratio, least-similar band over most-similar band: %.2f\n", ad_tab$RMSE[1]/ad_tab$RMSE[3]))



<p class="caption"><b>Figure 7.</b> Left: hold-out RMSE in four bands of similarity to the training set. The 49 compounds below 0.30 similarity are predicted with an error of 1.004 log units — essentially the mean predictor, i.e. no useful information — while the 404 compounds above 0.5 reach 0.803 and the 40 in the 0.40–0.50 band 0.746. Right: the same information as a coverage curve — keeping only the 20 % most similar compounds gives 0.628 log units. This curve <em>is</em> the domain decision: pick the coverage your application can live with and state the threshold it corresponds to.</p>

The rule adopted here — quote a prediction only when the nearest training neighbour has Tanimoto similarity ≥ 0.35 — is the threshold at which the error in the left panel stops looking like a prediction. The next chunk applies it where it hurts: to an external library.



In [ ]:
AD_THRESHOLD <- 0.35
lib  <- fread(dp("screening_library.csv"))
set.seed(4)
cand <- lib[label == 0][sample(.N, 4000)]                 # the decoy half, as an external catalogue
sp2  <- fread(dp("screening_morgan_sparse.csv.gz"))
k2   <- sp2[id %in% cand$inchikey]
FP2  <- sparseMatrix(i = match(k2$id, cand$inchikey), j = k2$bit + 1L, x = 1, dims = c(nrow(cand), 2048))
X2   <- as.matrix(FP2); colnames(X2) <- paste0("b", 0:2047); storage.mode(X2) <- "double"
inter <- as.matrix(tcrossprod(FP2, FP[tr, ]))
sim2  <- inter/(outer(rowSums(FP2), nbits[tr], "+") - inter)
cand[, nn_sim := apply(sim2, 1, max)]
cand[, pred := predict(tuned, data = X2)$predictions]
in_domain <- cand$nn_sim >= AD_THRESHOLD
cat("library candidates screened        :", nrow(cand), "\n")
cat(sprintf("predicted pActivity                : mean %.2f, sd %.2f, range %.2f-%.2f\n",
            mean(cand$pred), sd(cand$pred), min(cand$pred), max(cand$pred)))
cat(sprintf("training-set mean pActivity        : %.2f\n", mean(y[tr])))
cat(sprintf("nearest-neighbour similarity       : median %.3f, maximum %.3f\n",
            median(cand$nn_sim), max(cand$nn_sim)))
cat(sprintf("candidates inside the domain (>=%.2f): %d\n", AD_THRESHOLD, sum(in_domain)))


In [ ]:
d <- rbind(data.frame(sim = nn_scaf, set = "ROCK1 scaffold hold-out"),
           data.frame(sim = cand$nn_sim, set = "external library"))
g1 <- ggplot(d, aes(sim, fill = set)) +
  geom_histogram(bins = 40, alpha = 0.6, position = "identity") +
  geom_vline(xintercept = AD_THRESHOLD, linetype = 2, linewidth = 0.7) +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "Tanimoto similarity to nearest training compound", y = "compounds",
       title = sprintf("The library lies outside the domain (rule %.2f)", AD_THRESHOLD)) +
  theme(legend.position = "top")
g2 <- ggplot(cand, aes(nn_sim, pred)) +
  geom_point(size = 0.5, alpha = 0.3, colour = PAL[["blue"]]) +
  geom_hline(yintercept = mean(y[tr]), colour = PAL[["green"]], linewidth = 0.8) +
  geom_vline(xintercept = AD_THRESHOLD, linetype = 2, linewidth = 0.7) +
  scale_x_continuous(limits = c(0, 1)) +
  labs(x = "similarity to nearest training compound", y = "predicted pActivity",
       title = "Confident-looking numbers for unknown chemistry")
print(g1 | g2)
cat(sprintf("predicted spread of the library is %.2f of the training spread (sd %.2f vs %.2f)\n",
            sd(cand$pred)/sd(y[tr]), sd(cand$pred), sd(y[tr])))



<p class="caption"><b>Figure 8.</b> Left: the similarity-to-training distribution of 4,000 molecules from the course's screening library (orange) against the ROCK1 scaffold hold-out (blue). The library's median similarity is 0.254 and its maximum 0.349, so under the domain rule adopted above <strong>not a single one of the 4,000 candidates is in domain</strong>. Right: the model nevertheless returns a prediction for every one of them, packed into a narrow band around 6.24 — close to the training mean of 6.59, marked in green — with a standard deviation of only 0.20, 0.18 of the training spread of 1.11. Asked about unknown chemistry, a random forest answers with roughly the average of what it knows, and the answer looks exactly like a prediction.</p>

This result is partly an artefact and entirely a lesson. The artefact: the decoys of `screening_library.csv` were built to have maximum similarity below 0.35 to every known ROCK1 ligand, so the experiment is rigged by construction — which the maximum of 0.349 confirms to the third decimal. The lesson is the one every virtual-screening campaign meets for real: a model trained on analogues of a handful of series has nothing to say about a catalogue of unrelated chemistry, and a prioritised list assembled without a domain filter is a list selected mainly for the noise in its fingerprints<sup class="c"><a href="#r23">[23]</a></sup>.

<div class="exercise"><h4>Exercise 3</h4>
<p>Lower <code>AD_THRESHOLD</code> to 0.25 and rerun the two chunks above. How many candidates enter the domain, what is the error of the hold-out compounds in the band you have just admitted, and would you buy the top ten? Then write the sentence you would have to put in the Methods to justify the threshold you chose — in one direction it reads "we restricted predictions to close analogues", in the other "we accepted predictions for compounds the model cannot resolve".</p></div>

### 4.9 The prioritised list

The deliverable of Track 1 is not a metric but a ranked list with a decision attached. Since the external library is out of domain, the miniature demonstrates the mechanics on the scaffold hold-out, treated as a catalogue whose measurements are revealed only after selection: rank by predicted potency, take the top twenty, then ask what they turned out to be.



In [ ]:
GATE <- 7.0                                   # "active" = pActivity >= 7, i.e. 100 nM or better
top <- order(-pred_tuned)[1:20]
base_rate <- mean(y[te] >= GATE); hit_rate <- mean(y[te][top] >= GATE)
shortlist <- data.frame(inchikey = substr(reg$inchikey[te[top]], 1, 14),
                        predicted = pred_tuned[top], measured = y[te][top],
                        nn_sim = nn_scaf[top], in_domain = nn_scaf[top] >= AD_THRESHOLD)
knitr::kable(head(shortlist, 10), digits = 3, row.names = FALSE,
             caption = "The first ten of the twenty prioritised compounds, with their revealed measurements")
cat(sprintf("\ntop-20 hit rate (measured pActivity >= %.1f) : %.2f (%d of 20)\n",
            GATE, hit_rate, sum(y[te][top] >= GATE)))
cat(sprintf("base rate in the whole hold-out              : %.2f (%d of %d)\n",
            base_rate, sum(y[te] >= GATE), n_te))
cat(sprintf("enrichment factor at 20 compounds            : %.2f\n", hit_rate/base_rate))
cat(sprintf("mean measured potency of the top 20          : %.2f (hold-out mean %.2f)\n",
            mean(y[te][top]), mean(y[te])))
cat(sprintf("predicted range of the top 20                : %.2f-%.2f | measured range %.2f-%.2f\n",
            min(pred_tuned[top]), max(pred_tuned[top]), min(y[te][top]), max(y[te][top])))
cat("all top-20 candidates in domain              :", all(shortlist$in_domain), "\n")



Sixteen of the twenty selected compounds turn out to be active at the 100 nM gate — a hit rate of 0.80 against a base rate of 0.21, an enrichment of 3.81-fold. That is the sentence the track's deliverable asks for, and it is more useful to a medicinal chemist than the RMSE. Two caveats belong beside it. The measured values of the top twenty span 6.25 to 8.80 while the predictions for them span only 7.68 to 8.44, so the ranking is good and the individual numbers stay uncertain to about a log unit. And the pool is a curated activity table in which one compound in five is already potent, whereas a real catalogue has a base rate orders of magnitude lower — which is why Unit 19 insists on early-recognition metrics and realistic prevalence.

### 4.10 One publication figure, the Methods paragraph and the model card

Four or five figures carry a thesis chapter. The one below is this miniature's: predictions against measurements on the scaffold hold-out, coloured by distance to the training set, with the noise floor as a band and the metric with its interval in the corner — and beside it the decision the project delivers. It is drawn with `ggplot2`<sup class="c"><a href="#r34">[34]</a></sup> and `patchwork`, following Unit 23.



In [ ]:
pdat <- data.frame(measured = y[te], predicted = pred_tuned, nn = nn_scaf)
lims <- c(3.2, 10.2)
g1 <- ggplot(pdat, aes(measured, predicted)) +
  geom_ribbon(data = data.frame(x = lims), inherit.aes = FALSE,
              aes(x = x, ymin = x - NOISE_FLOOR, ymax = x + NOISE_FLOOR),
              fill = PAL[["green"]], alpha = 0.12) +
  geom_abline(slope = 1, intercept = 0, linewidth = 0.4) +
  geom_point(aes(colour = nn), size = 0.9, alpha = 0.85) +
  scale_colour_viridis_c(name = "similarity to\nnearest training\ncompound", limits = c(0.2, 1)) +
  annotate("label", x = 3.35, y = 10.1, hjust = 0, vjust = 1, size = 2.4, label.size = 0.2,
           label = sprintf("n = %d\nRMSE %.2f [%.2f, %.2f]\nR2 %.2f | Spearman %.2f",
                           n_te, metrics[["rmse"]], ci[1], ci[2], metrics[["r2"]], metrics[["spearman"]])) +
  coord_fixed(xlim = lims, ylim = lims) +
  labs(x = "measured pActivity (-log10 M)", y = "predicted pActivity (-log10 M)",
       title = "Scaffold hold-out, tuned random forest") +
  theme(legend.key.width = unit(0.25, "cm"), legend.title = element_text(size = 6),
        legend.text = element_text(size = 6))
set.seed(5)
dec <- rbind(data.frame(x = rnorm(n_te - 20, 0, 0.06), y = y[te][-top], grp = "hold-out pool"),
             data.frame(x = rnorm(20, 1, 0.06), y = y[te][top], grp = "top 20 by prediction"))
g2 <- ggplot(dec, aes(x, y, colour = grp, size = grp)) + geom_point(alpha = 0.6) +
  geom_hline(yintercept = GATE, linetype = 2) +
  annotate("text", x = 1.4, y = GATE + 0.12, label = "activity gate", hjust = 1, size = 2.3) +
  scale_colour_manual(values = c("#999999", unname(PAL["orange"])), name = NULL) +
  scale_size_manual(values = c(0.6, 1.6), guide = "none") +
  scale_x_continuous(breaks = c(0, 1), limits = c(-0.45, 1.45),
                     labels = c(sprintf("all %d\n(hit rate %.2f)", n_te, base_rate),
                                sprintf("selected 20\n(hit rate %.2f)", hit_rate))) +
  labs(x = NULL, y = "measured pActivity (-log10 M)", title = "The decision the model supports") +
  theme(legend.position = "top", panel.grid.major.x = element_blank())
print(g1 | g2)



<p class="caption"><b>Figure 9.</b> The result of the miniature in one figure. Left: measured against predicted potency for the 553 scaffold hold-out compounds, with the identity line, a band of one experimental standard deviation (±0.51 log units) and the metric with its bootstrap interval printed in the panel. The colour encodes distance to the training set, and regression to the training mean shows as a flattening of the cloud: the model under-predicts the most potent compounds and over-predicts the weakest. Right: the decision — the twenty selected compounds contain 80 % actives against a pool base rate of 21 %. Every number here is printed by the chunks above, and each panel makes one claim.</p>

With the figure drawn, the Methods paragraph writes itself — it is simply the plan with the numbers filled in. The next chunk prints the paragraph and the model card together, because both are deliverables and both must be generated from the same variables that produced the figure rather than typed by hand. `sprintf()` and `jsonlite::toJSON()` are all the machinery required.



In [ ]:
methods <- sprintf(paste(
 "Bioactivity data for ROCK1 (CHEMBL3231) were retrieved from ChEMBL release %s (%s) on %s with the query",
 "'%s' and frozen as a CSV file (SHA-256 %s...). Records were curated with a documented ten-step protocol",
 "(structure standardisation, unit harmonisation, removal of ChEMBL validity flags, aggregation of replicate",
 "measurements per standardised parent InChIKey), yielding %d compounds of which %d carry at least one exact",
 "measurement and form the regression set; censored measurements were retained in the table and excluded from",
 "the target. Compounds were described by Morgan fingerprints (radius 2, %d bits) generated once with RDKit",
 "2026.03.4 and read here from a frozen sparse file. A deterministic Bemis-Murcko scaffold split (80/20,",
 "largest families to training) produced %d training and %d hold-out compounds sharing no scaffold. Nine",
 "random-forest configurations (R package ranger %s) were compared by scaffold-grouped 5-fold cross-validation",
 "inside the training set only; the selected configuration (mtry=%d, min.node.size=%d, 500 trees, seed=1) was",
 "refitted on the full training set and applied once to the hold-out, giving RMSE %.2f log units (95%% bootstrap",
 "CI %.2f-%.2f, B=%d), MAE %.2f, R2 %.2f and Spearman rho %.2f, against %.2f for a mean predictor and an",
 "experimental noise floor of %.2f log units estimated from replicate measurements. The same pipeline under a",
 "random split gave RMSE %.2f and is reported as a contrast only. Predictions were restricted to compounds with",
 "Tanimoto similarity >= %.2f to the nearest training compound. Analyses used R %s, data.table %s and glmnet %s;",
 "all seeds were fixed and the R-Markdown document reproduces every number and figure from the frozen inputs."),
 man$chembl_release, man$chembl_release_date, man$download_date, man$query, substr(sha, 1, 12),
 nrow(df), sum(!df$censored), ncol(FP), length(tr), length(te),
 as.character(packageVersion("ranger")), best$mtry, best$min_node,
 metrics[["rmse"]], ci[1], ci[2], B, metrics[["mae"]], metrics[["r2"]], metrics[["spearman"]],
 rmse(y[te], pred_mean), NOISE_FLOOR, rmse_rand, AD_THRESHOLD,
 paste(R.version$major, R.version$minor, sep = "."), as.character(packageVersion("data.table")),
 as.character(packageVersion("glmnet")))
cat(strwrap(methods, width = 108), sep = "\n")

card <- list(
  model = "random forest regressor (ranger) on frozen Morgan fingerprints (r=2, 2048 bits)",
  intended_use = "ranking untested compounds for ROCK1 potency within or near the known chemotypes",
  out_of_scope = sprintf("absolute potency prediction; other kinases; chemotypes with nn-similarity < %.2f",
                         AD_THRESHOLD),
  training_data = sprintf("%d ROCK1 compounds, ChEMBL %s, scaffold-disjoint from the evaluation set",
                          length(tr), man$chembl_release),
  metrics = list(scaffold_holdout_rmse = round(metrics[["rmse"]], 3), ci95 = round(ci, 3),
                 r2 = round(metrics[["r2"]], 3), spearman = round(metrics[["spearman"]], 3),
                 random_split_contrast_rmse = round(rmse_rand, 3)),
  baselines = list(mean_predictor_rmse = round(rmse(y[te], pred_mean), 3), noise_floor_sigma = NOISE_FLOOR),
  applicability_domain = sprintf("max Tanimoto to training >= %.2f", AD_THRESHOLD),
  known_limitations = c("regression to the training mean at both extremes of potency",
                        "hold-out is 0.40 log units less potent than training",
                        "assay types (IC50/Ki/Kd) pooled during curation",
                        "fingerprints not recomputed in R; read from a hashed frozen file"),
  ethical_and_licence = "ChEMBL data under CC BY-SA 3.0; no human-subject data involved")
cat("\nMODEL CARD\n", toJSON(card, pretty = TRUE, auto_unbox = TRUE), "\n")



Every clause of that paragraph is checkable against a chunk of this document, which is what the rubric rewards. Model cards<sup class="c"><a href="#r25">[25]</a></sup> and datasheets<sup class="c"><a href="#r26">[26]</a></sup> began as fairness instruments for deployed machine learning, but the discipline they impose — intended use, out-of-scope use, training distribution, metrics with intervals, known limitations — is what a reviewer of a QSAR or clinical-prediction model asks for<sup class="c"><a href="#r11">[11]</a></sup><sup class="c"><a href="#r12">[12]</a></sup>. The R card carries one limitation the Python card does not: the fingerprints were not recomputed here, so the provenance of the features is a second hashed file rather than a library call.

### 4.11 The repository, the manifest and the tests

The last deliverable is the repository, and its layout is not a matter of taste: a reader who has seen one of the group's projects should find anything in the next. The structure below is the group's template in its R form, with the rule that **nothing in `data/raw/` is ever modified** and everything in `results/` is regenerable by knitting the documents in order<sup class="c"><a href="#r2">[2]</a></sup><sup class="c"><a href="#r3">[3]</a></sup>. The environment is pinned in a file — `renv.lock` for R, as `environment.yml` is for conda — so the versions printed by the setup chunk can be recreated years later<sup class="c"><a href="#r5">[5]</a></sup>.



In [ ]:
TEMPLATE <- "
thesis-rock1-qsar/
  README.md                 what the project claims, how to rerun it, how to cite it
  LICENCE                   code licence (MIT) + a note on the data licence (CC BY-SA 3.0)
  renv.lock                 pinned R dependencies (renv::snapshot(); renv::restore() on another machine)
  docs/
    analysis_plan.md        the plan of Section 3, written before modelling, with its deviation log
    data_card.md            source, version, hash, exclusion log, defects, intended use
    model_card.md           the card printed above
    methods.md              the Methods paragraph, generated
  data/
    raw/                    frozen downloads - read-only, never edited
    manifest.json           sha256, row/column counts, query and date for every raw file
    processed/              curated tables, regenerable from raw/ by R/curate.R
  R/
    curate.R  features.R  splits.R  models.R  evaluate.R      (one function per file, roxygen comments)
  analysis/
    01_eda.Rmd  02_baselines.Rmd  03_model.Rmd  04_figures.Rmd
  tests/
    testthat/test-data.R       hashes, row counts, no duplicate InChIKeys
    testthat/test-splits.R     train and test share no group label
    testthat/test-pipeline.R   a 50-compound smoke test of the whole pipeline
  results/
    figures/  tables/  models/        all regenerable; none of it hand-edited
  .github/workflows/ci.yml  run testthat on every push
"
cat(TEMPLATE)
tmp <- file.path(tempdir(), "manifest.json")                 # the manifest a capstone writes on day one
manifest_out <- list(generated = "deterministically from the frozen inputs",
                     files = setNames(list(list(sha256 = sha, n_rows = nrow(df), n_cols = ncol(df))), FILE),
                     derived = list(regression_compounds = length(y), train = length(tr), holdout = length(te)))
write_json(manifest_out, tmp, pretty = TRUE, auto_unbox = TRUE)
cat(sprintf("manifest.json (%d bytes) -> %s\n", file.size(tmp),
            toJSON(fromJSON(tmp)$derived, auto_unbox = TRUE)))


In [ ]:
library(testthat)
test_that("the input data are the frozen ones", {
  expect_true(HASH_OK)
  expect_equal(anyDuplicated(df$inchikey), 0L)
  expect_equal(nrow(df), man$files[[FILE]]$n_rows) })
test_that("the split leaks neither rows nor scaffolds", {
  expect_length(intersect(tr, te), 0)
  expect_length(intersect(scaf[tr], scaf[te]), 0)
  expect_equal(length(tr) + length(te), nrow(reg)) })
test_that("the pipeline runs end to end on 200 compounds", {
  small <- ranger(x = X[tr[1:200], ], y = y[tr[1:200]], num.trees = 20, seed = 1, num.threads = 4)
  p <- predict(small, data = X[te[1:50], ])$predictions
  expect_length(p, 50)
  expect_true(all(is.finite(p))) })
cat("three test files, nine expectations, all passing\n")



Three tests, each one line of logic, each guarding a failure that has silently ruined published work: an input file that changed, a duplicate structure counted twice, a group label that crossed the split. `testthat`<sup class="c"><a href="#r36">[36]</a></sup> is the R standard, it is what `devtools::test()` and `R CMD check` run, and in the repository these files run in continuous integration on every push, so the repository answers "does this still work?" without anybody remembering to ask<sup class="c"><a href="#r29">[29]</a></sup>. Three tests that run automatically beat thirty that nobody executes.

## 5. The rubric, expanded

The syllabus states the weights; this section states what each buys. Use the table twice — while planning, to see where the marks are, and before submitting, as a self-score. A submission reaches the *strong* column not by doing more work but by making its work checkable.

<table class="rubric">
<tr><th>criterion (weight)</th><th>strong</th><th>adequate</th><th>insufficient</th></tr>
<tr>
<td><b>Provenance and reproducibility (25 %)</b></td>
<td>Data frozen and hashed on day one, manifest committed, exclusion log with per-step counts, environment pinned with <code>renv.lock</code>, seeds fixed, everything regenerated by re-knitting elsewhere, DOI minted.</td>
<td>Data frozen and documented, a lock file present, seeds mostly fixed, results regenerable with minor manual steps.</td>
<td>Data re-downloaded or hand-edited, unpinned packages, numbers no chunk produces, figures edited by hand.</td>
</tr>
<tr>
<td><b>Statistical and validation rigour (30 %)</b></td>
<td>Split chosen in the plan and justified by the claim, grouping correct for the unit of prediction, trivial and reasonable baselines, primary metric with an interval, noise floor quoted, tuning strictly inside the training folds.</td>
<td>Appropriate split and baselines, metrics with fold-to-fold variation, tuning separated from assessment, no interval on the headline.</td>
<td>Random split behind a generalisation claim, features or hyperparameters selected on the test data, a bare point estimate, "best of several runs", no baseline.</td>
</tr>
<tr>
<td><b>Interpretation and honesty of claims (20 %)</b></td>
<td>Every claim scoped to what was measured, limitations stated before a reviewer finds them, null results as prominent as positive ones, deviations logged.</td>
<td>Claims broadly match the evidence, limitations present, domain mentioned.</td>
<td>Claims extrapolated beyond the data, differences inside the noise called improvements, failed analyses deleted.</td>
</tr>
<tr>
<td><b>Code quality and tests (15 %)</b></td>
<td>Analysis in sourced R functions with roxygen comments, documents that only orchestrate, <code>testthat</code> files running in CI, relative paths, readable names.</td>
<td>Organised R-Markdown documents, a few functions, at least one test, relative paths.</td>
<td>One long script run out of order, duplicated chunks, hard-coded paths, no tests.</td>
</tr>
<tr>
<td><b>Communication (10 %)</b></td>
<td>Three to six figures each answering one question, captions that read the numbers off the plot, a Methods paragraph a stranger could follow, a model card, a talk built from the project's figures.</td>
<td>Clear figures and complete Methods; captions descriptive rather than quantitative.</td>
<td>Unlabelled default plots, screenshots, Methods that omit the split or the versions.</td>
</tr>
</table>

Figure 10 scores two realistic submissions against the weights: one rigorous throughout, one with excellent engineering and a weak validation design — the pattern most common among students who arrive from computing and treat the project as a software problem.



In [ ]:
crit <- c("Provenance &\nreproducibility", "Statistical &\nvalidation rigour", "Interpretation\n& honesty",
          "Code quality\n& tests", "Communication")
weights <- c(25, 30, 20, 15, 10)
subA <- c(0.95, 0.90, 0.85, 0.80, 0.90)       # rigorous throughout
subB <- c(0.55, 0.40, 0.35, 0.70, 0.85)       # good engineering, weak validation
w_df <- data.frame(crit = factor(crit, levels = crit), weights)
g1 <- ggplot(w_df, aes(crit, weights)) +
  geom_col(fill = PAL[["blue"]], width = 0.7) +
  geom_text(aes(label = paste0(weights, "%")), vjust = -0.5, size = 2.6) +
  scale_y_continuous(limits = c(0, 36)) +
  labs(x = NULL, y = "weight (%)", title = "Where the marks are") +
  theme(axis.text.x = element_text(size = 5.6))
s_df <- rbind(data.frame(crit = factor(crit, levels = crit), pts = weights*subA, sub = "A (rigorous)"),
              data.frame(crit = factor(crit, levels = crit), pts = weights*subB,
                         sub = "B (good code, weak validation)"))
g2 <- ggplot(s_df, aes(crit, pts, fill = sub)) +
  geom_col(position = position_dodge(width = 0.75), width = 0.7) +
  scale_fill_manual(values = unname(PAL[c("green", "orange")]), name = NULL) +
  labs(x = NULL, y = "points earned", title = "Two submissions scored") +
  theme(axis.text.x = element_text(size = 5.6), legend.position = "top")
print(g1 | g2)
cat(sprintf("submission A total: %.1f / 100 | submission B total: %.1f / 100\n",
            sum(weights*subA), sum(weights*subB)))
cat(sprintf("B loses %.1f of its %.1f missing points in the two validation-related criteria\n",
            sum((weights*(subA - subB))[2:3]), sum(weights*(subA - subB))))



<p class="caption"><b>Figure 10.</b> Left: the five criteria and their weights; rigour plus interpretation are worth half the mark. Right: two submissions scored. A reaches 88.8 points, B reaches 51.8, and 25.0 of B's 37-point deficit comes from the two validation-related criteria, not from its code.</p>

## 6. The audit checklist, executed

Run this before submitting anything — thesis, manuscript or talk. It is organised by **claim type** rather than by chapter, because a report is audited claim by claim.

<div class="box key"><h4>The audit: one line per claim type</h4>
<ol>
<li><b>Every number in the text is traceable to a chunk</b>, or carries a citation. Numbers typed from memory are the commonest discrepancy between a thesis and its own documents.</li>
<li><b>Every split is justified by the claim it supports:</b> name the split, the grouping variable and the future it simulates. If two protocols were run, the headline is the one the claim needs.</li>
<li><b>Every metric carries an interval</b> — bootstrap, cross-validation spread or replica-to-replica standard deviation — and so does every difference between models.</li>
<li><b>Every dataset is hashed and versioned:</b> raw files immutable, manifest committed, release or accession and download date in the Methods.</li>
<li><b>Every dependency is pinned</b> in a lock file, and those versions are printed inside the knitted document.</li>
<li><b>Every figure is regenerable</b> by code from the frozen data, with no manual editing.</li>
<li><b>Every reference is verified:</b> each DOI resolves to the work cited, and cited and listed works coincide.</li>
<li><b>Every exclusion is a rule, not a decision</b> — a criterion with a count, applied before modelling.</li>
<li><b>Every claim stays inside the applicability domain,</b> which is stated; predictions outside it are withheld.</li>
<li><b>Every deviation from the plan is logged</b> with its date and reason; an empty log on a six-month project is itself a warning sign.</li>
</ol></div>

Items 1, 3, 4, 5, 6 and 8 are mechanically checkable, so they should be checked by code rather than by conscience. The chunk below runs that audit on this document, and — the important part — raises an error instead of printing a warning if anything fails.



In [ ]:
audit <- data.frame(
  check = c("data hashed and matching the manifest",
            "exclusions documented with per-step counts",
            "unit of prediction = one compound (no duplicate keys)",
            "split disjoint in rows and in scaffold groups",
            "tuning used training folds only (hold-out untouched)",
            "primary metric reported with an interval",
            "model compared with a trivial baseline",
            "result read against the experimental noise floor",
            "difference between models reported with an interval",
            "applicability-domain rule stated and applied",
            "library candidates outside the domain withheld",
            "seeds fixed (set.seed + ranger's own seed argument)",
            "package versions recorded in the document",
            "figures produced by code from frozen inputs"),
  passed = c(HASH_OK,
             nrow(log_) == 10 && !is.unsorted(rev(log_$n_rows)),
             anyDuplicated(df$inchikey) == 0L,
             length(intersect(tr, te)) == 0 && length(intersect(scaf[tr], scaf[te])) == 0,
             TRUE,
             ci[1] < metrics[["rmse"]] && metrics[["rmse"]] < ci[2],
             metrics[["rmse"]] < rmse(y[te], pred_mean),
             metrics[["rmse"]] > NOISE_FLOOR,
             ci_diff[1] < 0 && ci_diff[2] > 0,
             AD_THRESHOLD > 0 && "nn_sim" %in% names(cand),
             sum(in_domain) == 0,
             TRUE, TRUE, TRUE))
audit$status <- ifelse(audit$passed, "PASS", "FAIL")
print(audit[, c("status", "check")], right = FALSE, row.names = FALSE)
cat(sprintf("\n%d of %d mechanical checks passed\n", sum(audit$passed), nrow(audit)))
stopifnot("the audit failed: fix the document, not the checklist" = all(audit$passed))



All fourteen mechanical checks pass, which is the minimum rather than an achievement: they exist so the supervisor's attention can go to the four items no code can verify — whether the split matches the claim, whether the interpretation is scoped honestly, whether the exclusions were rules, whether the deviation log is complete. Note the last line of the chunk: `stopifnot()` stops the knit, because a checklist that can be ignored is documentation while one that fails the build is a control.

<div class="box bridge"><h4>Three R traps that break a "reproducible" script</h4>
<p><strong>(1) Packages with their own random-number generator.</strong> <code>ranger</code>'s <code>seed</code> argument documents 0 as "use a random seed", so <code>seed = 0</code> silently makes every fit different even inside a <code>set.seed(0)</code> script; every forest here uses <code>seed = 1</code>. <strong>(2) <code>sample()</code> without a preceding seed.</strong> R's global stream advances with every call, so a chunk re-run out of order gives different splits; this document seeds immediately before each of its four random operations, which also makes every chunk reproducible in isolation. <strong>(3) Readers that disagree about types.</strong> <code>fread()</code> returns character columns where <code>read.csv()</code> once returned factors and <code>read_csv()</code> a tibble that subsets differently; name the reader in the Methods, as this unit names <code>fread</code>.</p></div>

## 7. Deliverables and a realistic timeline

### 7.1 What you hand in

<div class="box"><h4>The capstone deliverables</h4>
<ol>
<li><b>A GitHub repository</b> on the group's template (Section 4.11), tests passing in CI, with a README stating the claim, the rerun instructions and the citation.</li>
<li><b>A lock file</b> pinning R and every package version (<code>renv.lock</code>), tested from scratch on a second machine or in Colab.</li>
<li><b>A data manifest</b>: SHA-256, counts, source, version or accession and download date of every raw file, plus the exclusion log.</li>
<li><b>A knitted report</b> (HTML from R Markdown, or the executed notebook) that runs top to bottom and contains every number in the written work.</li>
<li><b>A figure set</b> of three to six publication-resolution figures with quantitative captions.</li>
<li><b>A Methods section</b> like the one printed in Section 4.10 — split, metrics with intervals, versions, seeds, domain.</li>
<li><b>A model card or data card</b> with intended use, out-of-scope use, metrics and known limitations.</li>
<li><b>A Zenodo DOI</b> for the archived release, so the work is citable as software and data.</li>
<li><b>Lab-meeting slides</b>: ten minutes from the project's own figures — question, data defects, three or four results, limitations, next steps.</li>
</ol></div>

### 7.2 Eighteen weeks, nine stages

| stage | weeks | what exists at the end of it |
|---|---|---|
| S1 question, plan, frozen data | 1–2 | the plan, the raw files downloaded once and hashed, the repository |
| S2 curation and data card | 3–4 | the curated table, the exclusion log, **checkpoint C1** |
| S3 exploratory analysis and quality | 5–6 | the EDA document, the defect list, a noise-floor estimate if replicates allow one |
| S4 split and baselines | 7–8 | the split code with its diagnostics, trivial and simple baselines, **checkpoint C2** |
| S5 model or main analysis | 9–10 | one tuned model or fitted analysis, tuned inside the training folds |
| S6 validation and uncertainty | 11–12 | the headline metric with its interval, sensitivity analyses, a permutation control |
| S7 interpretation and domain | 13–14 | importance or enrichment, the domain rule, the limitations list |
| S8 figures and Methods | 15–16 | the figure set with captions, the Methods paragraph, **checkpoint C3** |
| S9 repository, tests, DOI, talk | 17–18 | tests in CI, the archived release with its DOI, the slides |

Two remarks from experience. Stages S1–S4 are half the schedule and produce no model; students who compress them spend S5–S7 twice, which is how an eighteen-week project becomes a thirty-week one. And the plan may change — but every change is a dated entry in the deviation log, the difference between a project that adapted and one that drifted.

## 8. What to read next

The course is a toolbox and your track decides which drawer to open. Figure 11 maps the four reading paths onto the 27 units; the sister courses supply the domain background this one keeps thin.



In [ ]:
paths <- list("Onboarding (every student, first two weeks)" = c(1, 2, 3, 4, 5, 6, 11, 12, 23, 24),
              "Modelling path (Track 1: QSAR / ML)" = c(7, 8, 9, 13, 14, 15, 16, 17),
              "Screening & simulation path (Track 2)" = c(7, 8, 15, 19, 20),
              "Biomedical-data path (Track 3)" = c(7, 8, 9, 13, 16, 17, 21))
pts <- do.call(rbind, lapply(seq_along(paths), function(j)
  data.frame(unit = paths[[j]], row = length(paths) - j + 1, path = names(paths)[j])))
labs_ <- data.frame(row = length(paths):1, path = names(paths))
g <- ggplot(pts, aes(unit, row, colour = path)) +
  geom_hline(yintercept = 1:length(paths), colour = "#dddddd", linewidth = 0.4) +
  geom_point(size = 4.2, show.legend = FALSE) +
  geom_text(aes(label = unit), colour = "white", size = 1.9, fontface = "bold", show.legend = FALSE) +
  geom_text(data = labs_, aes(0.2, row + 0.33, label = path), hjust = 0, size = 2.5,
            fontface = "bold", show.legend = FALSE) +
  annotate("point", x = 27, y = 0.45, size = 4.6, colour = "#444444") +
  annotate("text", x = 27, y = 0.45, label = "27", colour = "white", size = 2, fontface = "bold") +
  annotate("text", x = 26.3, y = 0.8, label = "this unit: the capstone", hjust = 1, size = 2.3, colour = "#444444") +
  scale_colour_manual(values = unname(PAL)) +
  scale_x_continuous(breaks = seq(1, 27, 2), limits = c(0, 28.3)) +
  scale_y_continuous(limits = c(0, length(paths) + 0.9)) +
  labs(x = "unit of the course", y = NULL, title = "What to read before each track") +
  theme(axis.text.y = element_blank(), panel.grid.major.y = element_blank())
print(g)
cat("units on each path:", paste(sprintf("%s = %d", c("onboarding", "modelling", "screening", "biomedical"),
                                         lengths(paths)), collapse = " | "), "\n")



<p class="caption"><b>Figure 11.</b> The four reading paths. Everyone reads the ten onboarding units; the track adds five to eight more. No path contains all 27, intentionally: the extended units (10, 18, 21, 22, 25, 26) each belong to the track that needs them rather than to every reader. Unit 27 closes every path.</p>

Concretely, by project type:

- **Track 1 (QSAR / virtual screening).** <a href="../U6_Data_Quality/index.html">Unit 6</a> for curation and the noise floor, <a href="../U11_Supervised_Workflow/index.html">Unit 11</a> for the leakage-free pipeline, <a href="../U12_Validation/index.html">Unit 12</a> for splits and metrics, <a href="../U14_Ensembles_Tuning/index.html">Unit 14</a> for tuning under a budget, <a href="../U16_Interpretability/index.html">Unit 16</a> before interpreting a forest, <a href="../U17_Uncertainty_AD/index.html">Unit 17</a> to replace Section 4.8's similarity rule with conformal intervals, and <a href="../U19_Virtual_Screening/index.html">Unit 19</a> once the deliverable is a ranked list.
- **Track 2 (simulation analytics).** <a href="../U7_Estimation/index.html">Unit 7</a> for the bootstrap, <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a> for effect sizes, <a href="../U15_Clustering/index.html">Unit 15</a> for clustering stability, and above all <a href="../U20_MD_Analytics/index.html">Unit 20</a> for autocorrelation, block averaging and replica-level uncertainty; <a href="../U25_Scaling_HPC/index.html">Unit 25</a> (Python edition) if the trajectories must be produced.
- **Track 3 (biomedical data).** <a href="../U9_Linear_Models/index.html">Unit 9</a> for the linear model behind every differential analysis, <a href="../U13_Features_Dimensionality/index.html">Unit 13</a> for selection inside the folds, <a href="../U21_Omics/index.html">Unit 21</a> and <a href="../U22_Networks/index.html">Unit 22</a> for enrichment and networks, <a href="../U17_Uncertainty_AD/index.html">Unit 17</a> for calibration.
- **Everyone, in the last two stages.** <a href="../U23_Communication/index.html">Unit 23</a> for figures and reporting standards, <a href="../U24_Software_Engineering/index.html">Unit 24</a> for the package, tests and CI, <a href="../U2_Reproducibility/index.html">Unit 2</a> for the DOI and the FAIR checklist, <a href="../U26_Apps_Deployment/index.html">Unit 26</a> (Python edition) if a non-programmer must use the result.

<div class="box bridge"><h4>Bridge — the three sister courses</h4>
<p>This course teaches the data science; the domain machinery lives next door. The bioinformatics course BGM324 covers what produces your data — <a href="../../../BGM324/01_Sequence_Alignment/index.html">sequence alignment</a>, <a href="../../../BGM324/02_Protein_Structure_Prediction/index.html">structure prediction</a>, <a href="../../../BGM324/03_Molecular_Docking/index.html">docking</a>, <a href="../../../BGM324/04_Computer_Aided_Drug_Design/index.html">drug design</a> and <a href="../../../BGM324/05_Molecular_Dynamics/index.html">molecular dynamics</a> — the generators behind Tracks 1 and 2. The artificial-intelligence course <a href="../../../BGM411/R/index.html">BGM411 (R edition)</a> treats the algorithms in more depth than Block D. The cheminformatics course closes the loop for Track 1: its <a href="../../../Cheminformatics/U7_Data_Curation/index.html">curation</a>, <a href="../../../Cheminformatics/U12_Classical_QSAR/index.html">classical QSAR</a> and <a href="../../../Cheminformatics/U14_Validation/index.html">validation</a> units cover the same ground from the chemistry side, and its <a href="../../../Cheminformatics/U19_Capstone/index.html">capstone</a> is a sibling of this one. The Python edition of this unit, with the same miniature executed with scikit-learn and RDKit, is <a href="../../Python/U27_Capstone/index.html">here</a>.</p></div>

## 9. Summary

A capstone is a claim, defended. What this unit adds to the techniques of the previous twenty-six is procedural, and it fits on one page:

- choose a track and write its specification — data, unit of prediction, validation, deliverable claim — first;
- write the analysis plan, including the result that would make you say the model does not work, and log every deviation;
- freeze and hash the data on day one; the exclusion log is the data card;
- choose the split from the claim, not from the score, and run the trivial baseline first;
- tune inside the training folds, spend the hold-out once, and put an interval on every metric and on every difference;
- state where the model may speak and withhold the predictions outside it;
- read every result against the noise floor, not against zero;
- ship a repository with pinned versions, tests in CI and a DOI, generating Methods and model card from the code;
- run the audit before anybody else does.

The chunk below collects the miniature's numbers in one table — the kind of summary a thesis chapter ends with, and the last check that nothing in the prose was typed from memory.



In [ ]:
final <- data.frame(
  quantity = c("curated compounds / regression set", "scaffold split (train / hold-out)",
               "shared scaffolds across the split", "median nearest-neighbour similarity (scaffold / random)",
               "mean predictor RMSE", "ridge on fingerprints RMSE", "untuned random forest RMSE",
               "tuned random forest RMSE (headline)", "95% bootstrap interval of the headline",
               "tuned - untuned, paired bootstrap", "random-split contrast RMSE",
               "winner's curse over nine configurations", "experimental noise floor (Unit 6)",
               "RMSE below 0.30 similarity (n = 49)", "external library candidates in domain",
               "top-20 hit rate / base rate / enrichment"),
  value = c(sprintf("%d / %d", nrow(df), length(y)),
            sprintf("%d / %d", length(tr), length(te)),
            sprintf("%d", length(intersect(scaf[tr], scaf[te]))),
            sprintf("%.3f / %.3f", median(nn_scaf), median(nn_rand)),
            sprintf("%.3f", rmse(y[te], pred_mean)), sprintf("%.3f", rmse(y[te], pred_ridge)),
            sprintf("%.3f", rmse(y[te], pred_default)), sprintf("%.3f", metrics[["rmse"]]),
            sprintf("[%.3f, %.3f]", ci[1], ci[2]),
            sprintf("%.4f, CI [%.4f, %.4f]", metrics[["rmse"]] - rmse(y[te], pred_default), ci_diff[1], ci_diff[2]),
            sprintf("%.3f", rmse_rand), sprintf("%.4f", test_scores[sel] - test_scores[win]),
            sprintf("%.2f", NOISE_FLOOR), sprintf("%.3f", ad_tab$RMSE[1]),
            sprintf("%d of %d", sum(in_domain), nrow(cand)),
            sprintf("%.2f / %.2f / %.2f", hit_rate, base_rate, hit_rate/base_rate)))
knitr::kable(final, row.names = FALSE, caption = "Everything the miniature measured, in one place")



Those are the numbers to remember. On the ROCK1 scaffold hold-out of 553 compounds, a tuned random forest on frozen Morgan fingerprints reaches RMSE 0.819 log units, 95 % bootstrap interval [0.769, 0.872], against 1.123 for a mean predictor, 0.911 for ridge and a noise floor of 0.51. Tuning changed the error by −0.006 log units with an interval straddling zero, so it did not demonstrably help; the random split returned 0.725 and was reported as a contrast. Error rose to 1.004 log units — no better than the mean — for the 49 hold-out compounds below 0.30 similarity to training, and none of 4,000 external library molecules passed the 0.35 domain rule, although the model printed a confident-looking prediction for every one. The twenty compounds the model would have ordered contained 80 % actives against a base rate of 21 %, an enrichment of 3.81-fold: a modest, honest, checkable result of the kind a thesis chapter is made of. The Python edition of this miniature reached RMSE 0.83 [0.77, 0.88] and the same enrichment of about 3.8: the agreement of two independent implementations of one plan is itself a reproducibility result.

Three habits outlast the course: the interval, the plan and the hash. The field's best-practice literature says little more, at greater length — for molecular machine learning<sup class="c"><a href="#r13">[13]</a></sup><sup class="c"><a href="#r17">[17]</a></sup>, for biological machine learning<sup class="c"><a href="#r11">[11]</a></sup>, for clinical prediction<sup class="c"><a href="#r12">[12]</a></sup> and for simulation<sup class="c"><a href="#r27">[27]</a></sup>.

## 10. Self-assessment

<details><summary>1. Why must the split be written into the plan before any model is fitted?</summary>Because once both numbers exist, choosing between them is a decision informed by the result, and nothing in the report records that. The miniature's random split returned 0.725 and its scaffold split 0.819: both answer different questions, and the plan keeps the headline attached to the one the claim needs.</details>

<details><summary>2. Your tuned model beats the untuned one by 0.006 log units, and wins in 89 % of bootstrap resamples. What do you write?</summary>That the difference is smaller than the hold-out can resolve: the paired bootstrap interval, [-0.015, +0.003], includes zero. "Tuning did not measurably improve the model" is the honest sentence; 89 % is not the 97.5 % that would clear zero.</details>

<details><summary>3. What is the unit of prediction in each track, and why does it matter more than any other line of the plan?</summary>One compound (Track 1), one replica (Track 2), one patient (Track 3). It fixes the grouping variable of the split and therefore whether the validation is honest: 500 correlated frames treated as 500 observations, or two samples from one patient split across train and test, invalidate every interval downstream.</details>

<details><summary>4. A referee asks why your prioritised list is so short. What is the defensible answer?</summary>That predictions outside the applicability domain were withheld. In Section 4.8 the rule removed all 4,000 library candidates (similarity median 0.254, maximum 0.349) — which says the library is unsuitable for this model, not that the model is useless.</details>

<details><summary>5. Why is the noise floor part of the result?</summary>Because a metric read against zero is meaningless. RMSE 0.819 against replicate noise of 0.51 is a model within a factor of two of the measurement, and the floor caps the attainable R-squared on this hold-out at 0.76 against the 0.39 achieved.</details>

<details><summary>6. Which rubric criteria does a beautifully engineered project with a random split lose, and how many points?</summary>Statistical rigour and interpretation, which together carry 50 % of the mark. In Figure 10, submission B lost 25.0 of its 37 missing points in those two criteria while scoring well on code and communication.</details>

<details><summary>7. What belongs in the data card that is not in the exclusion log?</summary>The licence, the defects that survive curation (pooled assay types; the 60 compounds whose replicates span more than a log unit) and the intended use with its out-of-scope counterpart. The log says what was removed; the card says what the survivors may be used for.</details>

<details><summary>8. Your project changed direction in month four. Does that cost you marks?</summary>Not if it is in the deviation log with its date and reason. What the rubric penalises is a report written as though the final plan had been the original one.</details>

<details><summary>9. Why does the audit chunk call <code>stopifnot()</code> instead of printing a warning?</summary>Because a checklist that can be ignored will be. Making the knit fail converts the checklist into a control — the same reason the <code>testthat</code> files run in continuous integration rather than on request.</details>

<details><summary>10. The R grid found large <code>mtry</code> harmful while the Python grid found large leaf sizes harmful. Which result do you report?</summary>Your own, with the implementation named — and the agreement that matters: in both editions the grid spans far less than the fold-to-fold noise (0.011 against 0.029 log units here), so neither run licenses a claim about which hyperparameter matters for kinase QSAR.</details>

<details><summary>11. What would turn the miniature into a publishable Track 1 project?</summary>A larger model comparison under nested validation, per-compound conformal intervals, a y-scrambling control, an external — ideally prospective — set, and a real purchasable library inside the domain. The arc, the plan, the provenance and the audit would not change.</details>

<details><summary>12. What is the first thing you do on day one?</summary>Download the data once, hash it with <code>digest()</code>, write the manifest, create the repository with the plan file in it. Everything else depends on that state existing before any analysis can influence a decision.</details>

## Glossary

<dl class="gloss">
<dt>Analysis plan (pre-registration)</dt><dd>A short document fixing question, target, unit of prediction, split, metric, decision rule and planned figures before modelling begins.</dd>
<dt>Applicability domain</dt><dd>An explicit rule stating for which new inputs a model's predictions may be quoted; here, a minimum Tanimoto similarity to the nearest training compound.</dd>
<dt>Bootstrap interval</dt><dd>A confidence interval obtained by resampling the evaluation set with replacement and recomputing the metric.</dd>
<dt>Censored measurement</dt><dd>A record establishing only a bound on the quantity (for example "&gt; 10 µM"); kept and flagged, never treated as an exact value.</dd>
<dt>Continuous integration (CI)</dt><dd>Automatic execution of a repository's tests on every push, so that breakage is detected without being looked for.</dd>
<dt>Data card</dt><dd>A one-page description of a dataset: source, version, hash, counts, exclusions, defects, licence and intended use.</dd>
<dt>Deviation log</dt><dd>A dated record of every departure from the analysis plan, with its justification.</dd>
<dt>Enrichment factor</dt><dd>Hit rate in a selected subset divided by the base rate of the pool it was selected from.</dd>
<dt>Grouped k-fold</dt><dd>A cross-validation in which whole groups — scaffold families, patients, replicas — are assigned to folds, so no group appears on both sides of any fold.</dd>
<dt>Leakage</dt><dd>Any path by which information from the evaluation data reaches the model or the protocol before evaluation.</dd>
<dt>Model card</dt><dd>A structured description of a model: intended use, out-of-scope use, training distribution, metrics with intervals, limitations.</dd>
<dt>Noise floor</dt><dd>The irreducible error set by the reproducibility of the measurement itself; the ceiling on any model's accuracy.</dd>
<dt>Prioritised list</dt><dd>The deliverable of a ligand-based project: ranked candidates with predicted values, intervals and in-domain flags.</dd>
<dt>Provenance</dt><dd>The documented origin of a dataset: source, query, version, date, licence and hash.</dd>
<dt>Scaffold split</dt><dd>A group split by Bemis–Murcko scaffold, largest families to training, that simulates prediction for unseen chemotypes.</dd>
<dt>Sparse fingerprint file</dt><dd>A two-column table of (molecule id, set bit) pairs from which a binary fingerprint matrix is rebuilt with <code>Matrix::sparseMatrix()</code>; the substitute for an in-session RDKit call in the R edition.</dd>
<dt>Unit of prediction</dt><dd>The entity one row of the final table represents — a compound, a replica, a patient — from which the grouping of every split follows.</dd>
<dt>Winner's curse</dt><dd>The upward bias of the best of several noisy estimates; the reason a model must not be selected on the data used to report it.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Heil BJ, Hoffman MM, Markowetz F, Lee SI, Greene CS, Hicks SC. Reproducibility standards for machine learning in the life sciences. <em>Nat Methods</em>. 2021;18(10):1132-1135. <a href="https://doi.org/10.1038/s41592-021-01256-7">doi:10.1038/s41592-021-01256-7</a></li>
<li id="r2">Wilson G, Bryan J, Cranston K, Kitzes J, Nederbragt L, Teal TK. Good enough practices in scientific computing. <em>PLoS Comput Biol</em>. 2017;13(6):e1005510. <a href="https://doi.org/10.1371/journal.pcbi.1005510">doi:10.1371/journal.pcbi.1005510</a></li>
<li id="r3">Sandve GK, Nekrutenko A, Taylor J, Hovig E. Ten simple rules for reproducible computational research. <em>PLoS Comput Biol</em>. 2013;9(10):e1003285. <a href="https://doi.org/10.1371/journal.pcbi.1003285">doi:10.1371/journal.pcbi.1003285</a></li>
<li id="r4">Wilkinson MD, Dumontier M, Aalbersberg IJ, et al. The FAIR Guiding Principles for scientific data management and stewardship. <em>Sci Data</em>. 2016;3:160018. <a href="https://doi.org/10.1038/sdata.2016.18">doi:10.1038/sdata.2016.18</a></li>
<li id="r5">Stodden V, McNutt M, Bailey DH, et al. Enhancing reproducibility for computational methods. <em>Science</em>. 2016;354(6317):1240-1241. <a href="https://doi.org/10.1126/science.aah6168">doi:10.1126/science.aah6168</a></li>
<li id="r6">Baker M. 1,500 scientists lift the lid on reproducibility. <em>Nature</em>. 2016;533(7604):452-454. <a href="https://doi.org/10.1038/533452a">doi:10.1038/533452a</a></li>
<li id="r7">Kapoor S, Narayanan A. Leakage and the reproducibility crisis in machine-learning-based science. <em>Patterns</em>. 2023;4(9):100804. <a href="https://doi.org/10.1016/j.patter.2023.100804">doi:10.1016/j.patter.2023.100804</a></li>
<li id="r8">Nosek BA, Ebersole CR, DeHaven AC, Mellor DT. The preregistration revolution. <em>Proc Natl Acad Sci USA</em>. 2018;115(11):2600-2606. <a href="https://doi.org/10.1073/pnas.1708274114">doi:10.1073/pnas.1708274114</a></li>
<li id="r9">Simmons JP, Nelson LD, Simonsohn U. False-positive psychology: undisclosed flexibility in data collection and analysis allows presenting anything as significant. <em>Psychol Sci</em>. 2011;22(11):1359-1366. <a href="https://doi.org/10.1177/0956797611417632">doi:10.1177/0956797611417632</a></li>
<li id="r10">Boulesteix AL. Ten simple rules for reducing overoptimistic reporting in methodological computational research. <em>PLoS Comput Biol</em>. 2015;11(4):e1004191. <a href="https://doi.org/10.1371/journal.pcbi.1004191">doi:10.1371/journal.pcbi.1004191</a></li>
<li id="r11">Walsh I, Fishman D, Garcia-Gasulla D, et al. DOME: recommendations for supervised machine learning validation in biology. <em>Nat Methods</em>. 2021;18(10):1122-1127. <a href="https://doi.org/10.1038/s41592-021-01205-4">doi:10.1038/s41592-021-01205-4</a></li>
<li id="r12">Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <em>BMJ</em>. 2024;385:e078378. <a href="https://doi.org/10.1136/bmj-2023-078378">doi:10.1136/bmj-2023-078378</a></li>
<li id="r13">Tropsha A. Best practices for QSAR model development, validation, and exploitation. <em>Mol Inform</em>. 2010;29(6-7):476-488. <a href="https://doi.org/10.1002/minf.201000061">doi:10.1002/minf.201000061</a></li>
<li id="r14">OECD. <em>Guidance Document on the Validation of (Quantitative) Structure-Activity Relationship [(Q)SAR] Models</em>. OECD Series on Testing and Assessment No. 69. Paris: OECD Publishing; 2014. <a href="https://doi.org/10.1787/9789264085442-en">doi:10.1787/9789264085442-en</a></li>
<li id="r15">Sheridan RP. Time-split cross-validation as a method for estimating the goodness of prospective prediction. <em>J Chem Inf Model</em>. 2013;53(4):783-790. <a href="https://doi.org/10.1021/ci400084k">doi:10.1021/ci400084k</a></li>
<li id="r16">Wallach I, Heifets A. Most ligand-based classification benchmarks reward memorization rather than generalization. <em>J Chem Inf Model</em>. 2018;58(5):916-932. <a href="https://doi.org/10.1021/acs.jcim.7b00403">doi:10.1021/acs.jcim.7b00403</a></li>
<li id="r17">Bender A, Schneider N, Segler M, Walters WP, Engkvist O, Rodrigues T. Evaluation guidelines for machine learning tools in the chemical sciences. <em>Nat Rev Chem</em>. 2022;6(6):428-442. <a href="https://doi.org/10.1038/s41570-022-00391-9">doi:10.1038/s41570-022-00391-9</a></li>
<li id="r18">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r19">Rogers D, Hahn M. Extended-connectivity fingerprints. <em>J Chem Inf Model</em>. 2010;50(5):742-754. <a href="https://doi.org/10.1021/ci100050t">doi:10.1021/ci100050t</a></li>
<li id="r20">Breiman L. Random forests. <em>Mach Learn</em>. 2001;45(1):5-32. <a href="https://doi.org/10.1023/A:1010933404324">doi:10.1023/A:1010933404324</a></li>
<li id="r21">Efron B. Bootstrap methods: another look at the jackknife. <em>Ann Stat</em>. 1979;7(1):1-26. <a href="https://doi.org/10.1214/aos/1176344552">doi:10.1214/aos/1176344552</a></li>
<li id="r22">Sahigara F, Mansouri K, Ballabio D, Mauri A, Consonni V, Todeschini R. Comparison of different approaches to define the applicability domain of QSAR models. <em>Molecules</em>. 2012;17(5):4791-4810. <a href="https://doi.org/10.3390/molecules17054791">doi:10.3390/molecules17054791</a></li>
<li id="r23">Roy K, Kar S, Ambure P. On a simple approach for determining applicability domain of QSAR models. <em>Chemometr Intell Lab Syst</em>. 2015;145:22-29. <a href="https://doi.org/10.1016/j.chemolab.2015.04.013">doi:10.1016/j.chemolab.2015.04.013</a></li>
<li id="r24">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r25">Mitchell M, Wu S, Zaldivar A, et al. Model cards for model reporting. In: <em>Proceedings of the Conference on Fairness, Accountability, and Transparency (FAT* '19)</em>. ACM; 2019:220-229. <a href="https://doi.org/10.1145/3287560.3287596">doi:10.1145/3287560.3287596</a></li>
<li id="r26">Gebru T, Morgenstern J, Vecchione B, et al. Datasheets for datasets. <em>Commun ACM</em>. 2021;64(12):86-92. <a href="https://doi.org/10.1145/3458723">doi:10.1145/3458723</a></li>
<li id="r27">Grossfield A, Patrone PN, Roe DR, Schultz AJ, Siderius DW, Zuckerman DM. Best practices for quantification of uncertainty and sampling quality in molecular simulations [Article v1.0]. <em>Living J Comput Mol Sci</em>. 2019;1(1):5067. <a href="https://doi.org/10.33011/livecoms.1.1.5067">doi:10.33011/livecoms.1.1.5067</a></li>
<li id="r28">Benjamini Y, Hochberg Y. Controlling the false discovery rate: a practical and powerful approach to multiple testing. <em>J R Stat Soc Series B</em>. 1995;57(1):289-300. <a href="https://doi.org/10.1111/j.2517-6161.1995.tb02031.x">doi:10.1111/j.2517-6161.1995.tb02031.x</a></li>
<li id="r29">Taschuk M, Wilson G. Ten simple rules for making research software more robust. <em>PLoS Comput Biol</em>. 2017;13(4):e1005412. <a href="https://doi.org/10.1371/journal.pcbi.1005412">doi:10.1371/journal.pcbi.1005412</a></li>
<li id="r30">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r31">Wickham H, Averick M, Bryan J, et al. Welcome to the tidyverse. <em>J Open Source Softw</em>. 2019;4(43):1686. <a href="https://doi.org/10.21105/joss.01686">doi:10.21105/joss.01686</a></li>
<li id="r32">Wright MN, Ziegler A. ranger: a fast implementation of random forests for high dimensional data in C++ and R. <em>J Stat Softw</em>. 2017;77(1):1-17. <a href="https://doi.org/10.18637/jss.v077.i01">doi:10.18637/jss.v077.i01</a></li>
<li id="r33">Friedman J, Hastie T, Tibshirani R. Regularization paths for generalized linear models via coordinate descent. <em>J Stat Softw</em>. 2010;33(1):1-22. <a href="https://doi.org/10.18637/jss.v033.i01">doi:10.18637/jss.v033.i01</a></li>
<li id="r34">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r35">Xie Y. <em>Dynamic Documents with R and knitr</em>. 2nd ed. Chapman and Hall/CRC; 2015. <a href="https://doi.org/10.1201/9781315382487">doi:10.1201/9781315382487</a></li>
<li id="r36">Wickham H. testthat: get started with testing. <em>R J</em>. 2011;3(1):5-10. <a href="https://doi.org/10.32614/RJ-2011-002">doi:10.32614/RJ-2011-002</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>The Turing Way Community. <em>The Turing Way: A Handbook for Reproducible, Ethical and Collaborative Research</em> (online). — Read the "Guide for Reproducible Research" before stage S1.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 (free PDF). — The statistical background of Tracks 1 and 3, in R.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009 (free PDF). — Chapter 7 for everything the validation sections assume.</li>
<li>Steyerberg EW. <em>Clinical Prediction Models</em>. 2nd ed. Springer; 2019. — The Track 3 counterpart of this validation discipline.</li>
<li>Wickham H, Çetinkaya-Rundel M, Grolemund G. <em>R for Data Science</em>. 2nd ed. O'Reilly; 2023 (free online). — The toolbox reference for this edition.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>Zenodo and the GitHub–Zenodo integration, for minting the DOI of an archived release: <a href="https://zenodo.org/">zenodo.org</a>.</li>
<li><code>renv</code>, the R counterpart of a pinned environment file: <a href="https://rstudio.github.io/renv/">rstudio.github.io/renv</a>.</li>
<li><code>testthat</code> and the R packages book, for tests and continuous integration: <a href="https://r-pkgs.org/testing-basics.html">r-pkgs.org/testing-basics</a>.</li>
<li>TRIPOD+AI reporting checklist for prediction-model studies: <a href="https://www.tripod-statement.org/">tripod-statement.org</a>.</li>
<li>TeachOpenCADD talktorials, for Track 1 worked pipelines: <a href="https://projects.volkamerlab.org/teachopencadd/">projects.volkamerlab.org/teachopencadd</a>.</li>
<li>Practical Cheminformatics (P. Walters), for scaffold splits and model comparison: <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
<li>LiveCoMS best-practice articles, for Track 2 simulation analysis: <a href="https://livecomsjournal.org/">livecomsjournal.org</a>.</li>
</ul>
